In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from the image
data = {
    "Year": list(range(2002, 2027)),
    "Jan": [None, 0.57, 2.78, 1.27, -0.85, 2.85, 5.09, 0.31, 0.87, 1.67, 1.37, 2.89, 0.72, 3.84, -1.20, 1.02, 5.43, 0.62, 1.80, -1.64, 0.53, 1.21, -0.72, 0.45, 3.76],
    "Feb": [None, 1.77, 0.61, 2.06, 0.96, -0.90, 5.28, 1.82, 0.68, 2.00, 2.05, 0.32, 2.78, 3.69, -3.69, 2.05, -3.74, 2.02, -0.51, 6.34, 0.24, 0.45, -0.93, 0.49, 2.38],
    "Mar": [None, -0.74, -2.00, -0.45, 0.28, 1.18, -0.67, 5.31, 1.03, 0.31, -0.69, 0.46, -3.04, 1.91, -0.98, -1.18, -1.07, 1.78, -4.46, -3.60, 4.71, -4.73, 3.10, -2.76, -1.86],
    "Apr": [1.68, -2.09, -2.52, 2.86, 1.03, 5.05, -5.44, 1.14, -0.31, 2.80, 0.29, 0.61, -2.27, -3.91, -0.68, 1.29, 0.24, 1.38, 0.01, 3.12, 5.37, 2.01, 3.37, -2.44, 4.55],
    "May": [2.42, 3.66, -1.14, 2.99, -0.80, 5.62, 2.76, 2.52, -0.13, -2.26, 1.58, 1.59, 2.17, 4.11, -0.69, 4.37, 4.44, 2.14, 5.81, -0.70, -3.29, 1.87, -2.01, -1.35, None],
    "Jun": [4.51, 2.43, 0.62, 6.22, -2.13, 1.73, 2.18, -0.36, -1.92, -3.28, -2.61, -3.20, 0.28, -2.04, 4.68, -0.97, -1.22, 4.37, 2.88, -2.34, 1.45, -1.10, 1.59, 3.03, None],
    "Jul": [3.37, -1.77, -1.21, 0.92, -1.29, 0.08, -1.82, 1.33, -0.89, 0.76, 3.08, 2.74, -0.84, 1.96, 3.63, 1.30, 0.74, 1.42, 2.13, -0.29, -1.21, -0.43, -1.42, 2.82, None],
    "Aug": [1.04, 1.43, 1.18, 0.41, 0.11, -7.33, 1.21, 1.18, 4.81, -0.97, 1.64, 2.08, 2.00, -5.15, -0.33, 4.90, 2.53, 1.85, 0.76, -2.25, 3.55, 0.79, 0.19, 1.59, None],
    "Sep": [4.72, 3.91, -0.23, 4.54, -0.45, 4.26, -2.20, 4.17, -0.88, 1.23, -0.53, 1.63, 1.19, -1.56, 0.67, -0.70, 1.56, -5.41, 0.14, 0.93, 4.25, 1.59, 2.41, 4.05, None],
    "Oct": [-1.79, 1.18, 2.24, -4.72, 1.26, 4.99, -0.75, -0.99, 0.76, 1.10, -1.86, 0.57, -3.87, -0.43, 0.04, 3.82, -7.56, -2.64, 0.34, 0.34, 0.43, -0.13, -0.64, 5.26, None],
    "Nov": [-0.27, 0.64, 6.97, 4.02, 3.71, -0.55, 3.09, 3.04, -2.65, -2.62, 0.55, 2.63, 2.57, 2.41, -2.77, 0.17, -4.19, 0.22, 6.19, -3.46, -4.40, -4.06, 5.64, 0.74, None],
    "Dec": [0.01, 3.55, 3.03, 2.58, 5.47, -1.94, -0.78, 0.55, 4.38, 1.16, 0.03, 3.26, 1.97, 0.38, 0.10, -1.65, 2.02, 0.86, 8.77, -0.39, 2.46, 1.23, 2.83, 1.71, None]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
# Convert month abbreviations to numeric strings (e.g., 'Jan' -> '01')
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Drops non-existent months (Jan-Mar 2002 and May-Dec 2026)
)

# Convert return values from percentages to floating fractions (e.g., 1.68 -> 0.0168)
df_final["Return"] = df_final["Return"] / 100

print(df_final)

         Return
Period         
2002-04  0.0168
2002-05  0.0242
2002-06  0.0451
2002-07  0.0337
2002-08  0.0104
...         ...
2025-12  0.0171
2026-01  0.0376
2026-02  0.0238
2026-03 -0.0186
2026-04  0.0455

[289 rows x 1 columns]


In [4]:
df_final.to_csv('./hf_returns/brummer.csv')

In [5]:
import pandas as pd

# 1. Reconstruct the raw grid data from the image (ordered reverse-chronologically as shown)
data = {
    "Year": list(range(2026, 2010, -1)),
    "Jan": [0.77, 2.96, 1.05, 0.57, 2.46, -0.45, 0.00, 3.50, 0.74, 2.18, -0.48, -1.56, 1.89, 1.21, 3.81, None],
    "Feb": [-0.42, 1.06, -0.20, -0.43, 0.81, 2.74, 1.17, 0.12, -2.57, -0.37, -3.09, 1.22, 4.44, -4.95, -4.30, None],
    "Mar": [-4.30, -1.74, 1.75, 0.47, 0.69, -0.65, 3.62, 0.72, -0.66, 0.72, -1.24, 0.92, -0.33, 3.32, -0.32, 1.93],
    "Apr": [3.09, 0.83, 0.78, 0.86, 2.69, 0.96, 3.21, 3.55, 0.48, -0.90, 0.71, 2.12, -2.94, 1.64, -1.77, 1.29],
    "May": [None, 1.46, 1.28, 1.04, -2.08, -0.23, 3.86, -0.17, -0.69, 0.56, 0.63, 2.42, 2.27, 1.45, 0.65, 0.73],
    "Jun": [None, 2.32, 0.26, -1.02, 1.32, -0.11, 2.40, 0.01, 0.67, -1.28, -0.29, 0.59, 1.69, 0.77, 1.94, -0.45],
    "Jul": [None, 0.54, -0.25, 0.02, -0.33, -0.17, 4.52, 2.29, 1.56, -0.08, 0.71, 0.63, 0.27, 1.76, 4.84, 2.87],
    "Aug": [None, 0.72, 0.59, 0.71, 0.62, 0.77, 2.57, 0.27, 0.11, 2.71, 2.26, 0.09, 0.95, 1.05, 1.22, -3.17],
    "Sep": [None, 1.13, 0.01, 0.37, 1.69, 2.98, -0.06, -1.35, 1.43, 0.46, 1.33, -1.18, 5.41, 4.67, 1.28, -4.01],
    "Oct": [None, 2.36, 1.20, -0.02, -0.11, -0.47, 1.74, -0.39, -2.91, 0.90, -0.32, -0.37, -3.43, 3.65, 1.26, -1.53],
    "Nov": [None, 2.43, 4.24, -0.97, -1.28, 1.47, 3.74, 0.57, -5.59, -0.97, -1.07, 0.53, 1.47, 2.83, -0.14, -3.36],
    "Dec": [None, 1.08, 1.91, 1.06, 3.41, 0.81, 2.96, 2.23, 0.86, 2.88, 0.86, 0.94, 2.06, 2.43, 4.19, 0.79]
}

df_raw = pd.DataFrame(data)

# 2. Flatten the matrix structure into a single sequence
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Standardize month representations
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a proper time-series PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final cleaning and formatting
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()     # Put the index back into forward chronological order (2011 to 2026)
    .dropna()         # Drops non-existent months (e.g., Jan/Feb 2011 and May-Dec 2026)
)

# Convert percentage integers/floats into raw decimal returns (e.g., 0.77 -> 0.0077)
df_final["Return"] = df_final["Return"] / 100

print(df_final)

         Return
Period         
2011-03  0.0193
2011-04  0.0129
2011-05  0.0073
2011-06 -0.0045
2011-07  0.0287
...         ...
2025-12  0.0108
2026-01  0.0077
2026-02 -0.0042
2026-03 -0.0430
2026-04  0.0309

[182 rows x 1 columns]


In [6]:
df_final.to_csv('./hf_returns/bam.csv')

In [7]:
import pandas as pd

# 1. Reconstruct the raw grid data from the image (ordered chronologically by year)
data = {
    "Year": [2023, 2024, 2025, 2026],
    "Jan": [1.83, 1.78, 3.08, 0.72],
    "Feb": [4.49, 1.48, 0.83, 0.42],
    "Mar": [5.07, 7.00, 0.78, 0.45],
    "Apr": [1.89, 1.30, 0.12, 0.32],
    "May": [-0.39, 1.85, 0.97, None],
    "Jun": [1.63, 0.70, 1.52, None],
    "Jul": [0.15, 1.26, 2.06, None],
    "Aug": [-0.38, 0.67, 1.57, None],
    "Sep": [-0.39, 0.82, 1.37, None],
    "Oct": [0.19, 1.13, 0.47, None],
    "Nov": [0.56, 3.96, 0.65, None],
    "Dec": [0.95, 4.00, 0.20, None]
}

df_raw = pd.DataFrame(data)

# 2. Flatten the wide matrix format into a single sequence
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Standardize month abbreviations to two-digit numeric strings
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a proper time-series PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final cleaning, sorting, and decimal conversion
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()     # Organizes chronologically from Jan 2023 onward
    .dropna()         # Drops the empty future months (May-Dec 2026)
)

# Convert percentage values to standard fractional returns (e.g., 1.83 -> 0.0183)
df_final["Return"] = df_final["Return"] / 100

print(df_final)

         Return
Period         
2023-01  0.0183
2023-02  0.0449
2023-03  0.0507
2023-04  0.0189
2023-05 -0.0039
2023-06  0.0163
2023-07  0.0015
2023-08 -0.0038
2023-09 -0.0039
2023-10  0.0019
2023-11  0.0056
2023-12  0.0095
2024-01  0.0178
2024-02  0.0148
2024-03  0.0700
2024-04  0.0130
2024-05  0.0185
2024-06  0.0070
2024-07  0.0126
2024-08  0.0067
2024-09  0.0082
2024-10  0.0113
2024-11  0.0396
2024-12  0.0400
2025-01  0.0308
2025-02  0.0083
2025-03  0.0078
2025-04  0.0012
2025-05  0.0097
2025-06  0.0152
2025-07  0.0206
2025-08  0.0157
2025-09  0.0137
2025-10  0.0047
2025-11  0.0065
2025-12  0.0020
2026-01  0.0072
2026-02  0.0042
2026-03  0.0045
2026-04  0.0032


In [8]:
df_final.to_csv('./hf_returns/m1.csv')

In [16]:
import pandas as pd

# 1. Reconstruct the raw data directly from the image
data = {
    "Date": [
        "4/30/2026", "3/31/2026", "2/27/2026", "1/30/2026", "12/31/2025", "11/30/2025", 
        "10/31/2025", "9/30/2025", "8/29/2025", "7/31/2025", "6/30/2025", "5/30/2025", 
        "4/30/2025", "3/31/2025", "2/28/2025", "1/31/2025", "12/31/2024", "11/30/2024", 
        "10/31/2024", "9/30/2024", "8/30/2024", "7/31/2024", "6/30/2024", "5/31/2024", 
        "4/30/2024", "3/29/2024", "2/29/2024", "1/31/2024", "12/29/2023", "11/30/2023", 
        "10/31/2023", "9/29/2023", "8/31/2023", "7/31/2023", "6/30/2023", "5/31/2023", 
        "4/30/2023", "3/31/2023", "2/28/2023", "1/31/2023", "12/31/2022", "11/30/2022", 
        "10/31/2022", "9/30/2022", "8/31/2022", "7/29/2022", "6/30/2022", "5/31/2022", 
        "4/29/2022", "3/31/2022", "2/28/2022", "1/31/2022"
    ],
    "Net_Return_SMA": [
        None, None, None, 11.12, -2.63, -0.19, -4.79, 12.08, 1.49, 0.06, 2.05, -4.85, 
        4.46, 4.10, 3.55, -3.93, -6.00, 0.02, -6.65, 8.17, 0.63, 2.83, -2.02, -4.96, 
        5.31, 2.69, 12.06, 0.10, 5.41, 9.83, -0.70, 0.23, 0.41, -0.20, -0.03, 2.22, 
        -1.05, 6.03, -4.20, 4.95, -0.16, -1.42, -3.18, 1.40, 8.74, 4.69, 2.45, 5.24, 
        3.28, 12.87, 5.07, -2.51
    ],
    "Net_Return_Founder_Class": [
        -3.66, -3.47, -3.09, None, None, None, None, None, None, None, None, None, 
        None, None, None, None, None, None, None, None, None, None, None, None, 
        None, None, None, None, None, None, None, None, None, None, None, None, 
        None, None, None, None, None, None, None, None, None, None, None, None, 
        None, None, None, None
    ]
}

# 2. Initialize DataFrame
df = pd.DataFrame(data)

# 3. Convert the Date strings into a clean pandas Datetime Index
df["Date"] = pd.to_datetime(df["Date"])
df = df.set_index("Date")

# 4. Sort chronologically (oldest date 2022-01-31 to newest 2026-04-30)
df = df.sort_index()

# 5. Convert return values from raw percentages to decimal fractions (e.g., 11.12 -> 0.1112)
df["Net_Return_SMA"] = df["Net_Return_SMA"] / 100
df["Net_Return_Founder_Class"] = df["Net_Return_Founder_Class"] / 100

df_concat = df.sum(axis=1)

df_concat

Date
2022-01-31   -0.0251
2022-02-28    0.0507
2022-03-31    0.1287
2022-04-29    0.0328
2022-05-31    0.0524
2022-06-30    0.0245
2022-07-29    0.0469
2022-08-31    0.0874
2022-09-30    0.0140
2022-10-31   -0.0318
2022-11-30   -0.0142
2022-12-31   -0.0016
2023-01-31    0.0495
2023-02-28   -0.0420
2023-03-31    0.0603
2023-04-30   -0.0105
2023-05-31    0.0222
2023-06-30   -0.0003
2023-07-31   -0.0020
2023-08-31    0.0041
2023-09-29    0.0023
2023-10-31   -0.0070
2023-11-30    0.0983
2023-12-29    0.0541
2024-01-31    0.0010
2024-02-29    0.1206
2024-03-29    0.0269
2024-04-30    0.0531
2024-05-31   -0.0496
2024-06-30   -0.0202
2024-07-31    0.0283
2024-08-30    0.0063
2024-09-30    0.0817
2024-10-31   -0.0665
2024-11-30    0.0002
2024-12-31   -0.0600
2025-01-31   -0.0393
2025-02-28    0.0355
2025-03-31    0.0410
2025-04-30    0.0446
2025-05-30   -0.0485
2025-06-30    0.0205
2025-07-31    0.0006
2025-08-29    0.0149
2025-09-30    0.1208
2025-10-31   -0.0479
2025-11-30   -0.0019
2025-12-

In [17]:
df_concat.to_csv('./hf_returns/arr.csv')

In [10]:
import pandas as pd

# 1. Reconstruct the fragmented tables into a unified chronological map
# Stripping out 'n/a' months (Jan-May 2019) and the summary 'Total' rows
data = {
    "Period": [
        # 2019
        "2019-06", "2019-07", "2019-08", "2019-09", "2019-10", "2019-11", "2019-12",
        # 2020
        "2020-01", "2020-02", "2020-03", "2020-04", "2020-05", "2020-06", 
        "2020-07", "2020-08", "2020-09", "2020-10", "2020-11", "2020-12",
        # 2021
        "2021-01", "2021-02", "2021-03", "2021-04", "2021-05", "2021-06", 
        "2021-07", "2021-08", "2021-09", "2021-10", "2021-11", "2021-12",
        # 2022
        "2022-01", "2022-02", "2022-03", "2022-04", "2022-05", "2022-06", 
        "2022-07", "2022-08", "2022-09", "2022-10", "2022-11", "2022-12",
        # 2023
        "2023-01", "2023-02", "2023-03", "2023-04", "2023-05", "2023-06", 
        "2023-07", "2023-08", "2023-09", "2023-10", "2023-11", "2023-12",
        # 2024
        "2024-01", "2024-02", "2024-03", "2024-04", "2024-05", "2024-06", 
        "2024-07", "2024-08", "2024-09", "2024-10", "2024-11", "2024-12",
        # 2025
        "2025-01", "2025-02", "2025-03", "2025-04", "2025-05", "2025-06", 
        "2025-07", "2025-08", "2025-09", "2025-10", "2025-11", "2025-12"
    ],
    "Return": [
        # 2019
        2.37, 2.61, 1.12, 2.81, 2.22, -0.36, 3.76,
        # 2020
        2.15, -2.06, -3.42, 1.96, 1.57, 1.49, 1.16, 1.47, 1.39, -0.08, -0.25, 0.70,
        # 2021
        1.89, -0.43, -1.40, 3.80, 1.40, 1.89, -0.38, 2.07, -0.37, -0.97, -1.51, -0.72,
        # 2022
        -2.58, 3.37, 1.44, 2.18, 0.11, 1.72, 1.66, 1.46, 0.23, -0.85, -1.17, 5.26,
        # 2023
        -1.73, -1.08, -2.53, -1.79, 0.13, 0.05, -0.17, 0.60, -1.25, -0.37, 0.04, 0.81,
        # 2024
        -0.21, 1.29, 2.50, 0.29, 2.37, 5.27, 2.16, 1.15, 0.12, 0.10, -0.25, 1.31,
        # 2025
        1.41, 4.49, 1.31, 1.23, 0.72, 1.38, 1.14, 3.32, -0.62, 2.67, -0.54, 1.56
    ]
}

# 2. Create the DataFrame
df = pd.DataFrame(data)

# 3. Establish a standard time-series PeriodIndex (YYYY-MM)
df["Period"] = pd.to_datetime(df["Period"]).dt.to_period("M")
df = df.set_index("Period")

# 4. Turn percentage integers into standard fractional returns (e.g., 2.37 -> 0.0237)
df["Return"] = df["Return"] / 100

print(df)

         Return
Period         
2019-06  0.0237
2019-07  0.0261
2019-08  0.0112
2019-09  0.0281
2019-10  0.0222
...         ...
2025-08  0.0332
2025-09 -0.0062
2025-10  0.0267
2025-11 -0.0054
2025-12  0.0156

[79 rows x 1 columns]


In [11]:
df.to_csv('./hf_returns/four_world.csv')

In [12]:
import pandas as pd

# 1. Reconstruct the raw grid data from the image (ordered reverse-chronologically)
data = {
    "Year": list(range(2026, 2016, -1)),
    "Jan": [4.44, 9.72, 3.75, -5.58, 0.77, -3.78, -0.33, 6.96, -1.47, 5.90],
    "Feb": [1.58, 4.01, 1.11, -3.15, 3.87, -5.87, 1.85, 3.48, 6.58, -1.04],
    "Mar": [5.24, 0.24, 5.37, -6.56, -4.44, -5.02, 2.62, 2.49, 1.71, 5.03],
    "Apr": [1.15, -3.35, 0.91, -7.88, 3.42, -9.14, -4.33, 0.37, 12.01, -3.07],
    "May": [None, -1.12, 2.42, 21.06, 7.43, -5.37, -0.39, 1.41, 12.99, 1.71],
    "Jun": [None, -7.70, 5.90, 11.98, 11.88, 3.94, -5.60, 0.75, 4.49, 2.96],
    "Jul": [None, -3.03, 2.09, 5.25, 12.44, 18.84, -5.43, -1.82, 3.20, -0.17],
    "Aug": [None, 2.39, 8.08, -7.52, 8.95, 6.10, 5.71, 4.29, -2.20, -0.17],
    "Sep": [None, 2.55, 4.87, 11.02, 7.76, 7.28, 3.17, -1.98, 2.99, 3.33],
    "Oct": [None, 5.06, 3.47, 2.95, 1.68, 9.55, 7.01, 3.32, 3.07, 9.79],
    "Nov": [None, 7.98, 2.41, 0.70, -1.99, 7.45, 9.84, 0.15, 4.87, -1.87],
    "Dec": [None, 3.84, 5.59, -1.47, -0.28, -0.17, 5.76, 6.85, 4.71, -0.17]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single sequential column
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Standardize month mapping
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a time-series PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Filter, sort, and normalize scale
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()     # Puts data into natural forward order (Jan 2017 -> Apr 2026)
    .dropna()         # Drops uncompleted months (May 2026 onwards)
)

# Convert return values from raw percentages to decimal fractions (e.g., 4.44 -> 0.0444)
df_final["Return"] = df_final["Return"] / 100

print(df_final)

         Return
Period         
2017-01  0.0590
2017-02 -0.0104
2017-03  0.0503
2017-04 -0.0307
2017-05  0.0171
...         ...
2025-12  0.0384
2026-01  0.0444
2026-02  0.0158
2026-03  0.0524
2026-04  0.0115

[112 rows x 1 columns]


In [13]:
df_final.to_csv('./hf_returns/vadantia.csv')

In [21]:
import pandas as pd

# Original data matrix from the screenshot
data = {
    2018: [None, None, None, None, None, None, None, None, None, None, 2.4, 2.8, 5.3],
    2019: [0.0, 11.0, 2.9, 27.3, 41.4, 0.8, -6.4, -5.1, -7.2, 4.5, -5.4, -3.8, 62.5],
    2020: [20.4, 0.7, -3.6, 11.2, 0.1, -3.3, 18.4, 5.0, -4.6, 12.3, 30.9, 22.3, 168.1],
    2021: [17.0, 23.9, -0.3, 5.5, 2.9, -10.4, 7.4, 16.3, -9.8, 18.0, -2.3, -9.7, 64.9],
    2022: [0.4, -11.0, 8.9, -6.6, 3.7, -3.2, 7.1, -5.2, -13.0, -1.0, -5.1, -3.2, -26.6],
    2023: [21.5, -7.7, 4.2, -3.3, -4.0, 1.5, -4.2, 0.4, -2.2, 14.3, 9.6, 10.5, 43.4],
    2024: [-1.2, 20.8, 8.4, -2.5, -0.6, -2.2, -0.2, -2.0, -0.3, -0.9, 18.3, -1.0, 38.8],
    2025: [-3.0, -3.5, -4.9, -0.1, 1.3, -2.2, 10.4, 3.0, -0.4, -4.3, -5.6, -5.6, -14.9],
    2026: [1.1, -0.4, -6.9, -2.4, None, None, None, None, None, None, None, None, -8.4]
}

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'YTD']
df = pd.DataFrame(data, index=months)

# 1. Exclude YTD row
df_months = df.loc[df.index != 'YTD']

# 2. Unstack matrix into long-form rows
df_long = df_months.unstack().reset_index()
df_long.columns = ['Year', 'Month', 'Return']

# 3. Map months to numerical values and build a Datetime format column
month_map = {m: i+1 for i, m in enumerate(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])}
df_long['Month_Num'] = df_long['Month'].map(month_map)
df_long['Date'] = pd.to_datetime(df_long['Year'].astype(str) + '-' + df_long['Month_Num'].astype(str) + '-01')

# Sort chronologically and keep relevant columns
df_ts = df_long.sort_values('Date').reset_index(drop=True)[['Date', 'Return']]

# 4. Trim leading and trailing empty periods
first_valid = df_ts['Return'].first_valid_index()
last_valid = df_ts['Return'].last_valid_index()
df_ts_clean = df_ts.loc[first_valid:last_valid].copy().reset_index(drop=True)

# Set the Date as index for time series features
df_ts_clean.set_index('Date', inplace=True)

df_ts_clean = df_ts_clean / 100
# Save the tidy format to CSV
# df_ts_clean.to_csv('monthly_returns_timeseries.csv')

print(df_ts_clean)

            Return
Date              
2018-11-01   0.024
2018-12-01   0.028
2019-01-01   0.000
2019-02-01   0.110
2019-03-01   0.029
...            ...
2025-12-01  -0.056
2026-01-01   0.011
2026-02-01  -0.004
2026-03-01  -0.069
2026-04-01  -0.024

[90 rows x 1 columns]


In [22]:
df_ts_clean.to_csv('./hf_returns/cambrian.csv')

In [23]:
import pandas as pd

# Extracting data from the new image
# Columns: JAN, FEB, MAR, APR, MAY, JUN, JUL, AUG, SEP, OCT, NOV, DEC, YEAR
data_new = {
    2021: [None, None, None, None, None, None, None, None, None, 1.2, 0.3, 0.9, 2.4],
    2022: [-1.6, -0.2, -1.7, 1.0, 0.6, -2.3, 2.4, -0.1, -0.8, -0.9, 0.8, 1.3, -1.5],
    2023: [1.0, -1.1, 0.3, 1.0, 0.2, 1.0, 2.3, 1.2, 1.8, 1.0, 3.0, 2.2, 14.8],
    2024: [-0.9, 3.3, 2.0, -1.5, 0.5, 3.8, 2.8, 0.9, 0.7, 1.0, 0.9, -1.8, 12.2],
    2025: [1.3, 2.5, -1.9, 0.8, 1.2, 1.4, 0.2, 1.7, 4.1, 2.6, -0.9, -1.6, 11.7],
    2026: [4.1, 1.3, 0.4, None, None, None, None, None, None, None, None, None, 5.8]
}

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'YEAR']
df_new = pd.DataFrame(data_new, index=months)

# 1. Exclude the annual 'YEAR' aggregate row
df_months = df_new.loc[df_new.index != 'YEAR']

# 2. Reshape/Unstack from matrix format to long format
df_long = df_months.unstack().reset_index()
df_long.columns = ['Year', 'Month', 'Return']

# 3. Create numerical month mappings and construct full Datetime index
month_map = {m: i+1 for i, m in enumerate(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])}
df_long['Month_Num'] = df_long['Month'].map(month_map)
df_long['Date'] = pd.to_datetime(df_long['Year'].astype(str) + '-' + df_long['Month_Num'].astype(str) + '-01')

# Sort chronologically and drop temporary columns
df_ts = df_long.sort_values('Date').reset_index(drop=True)[['Date', 'Return']]

# 4. Filter out leading and trailing missing data to isolate the true timeline
first_valid = df_ts['Return'].first_valid_index()
last_valid = df_ts['Return'].last_valid_index()
df_ts_clean = df_ts.loc[first_valid:last_valid].copy().reset_index(drop=True)

# Set Date as the formal index for your time-series analysis
df_ts_clean.set_index('Date', inplace=True)

df_ts_clean = df_ts_clean /100
# Save to a new CSV file
# df_ts_clean.to_csv('monthly_returns_timeseries_2.csv')

print(df_ts_clean)

            Return
Date              
2021-10-01   0.012
2021-11-01   0.003
2021-12-01   0.009
2022-01-01  -0.016
2022-02-01  -0.002
2022-03-01  -0.017
2022-04-01   0.010
2022-05-01   0.006
2022-06-01  -0.023
2022-07-01   0.024
2022-08-01  -0.001
2022-09-01  -0.008
2022-10-01  -0.009
2022-11-01   0.008
2022-12-01   0.013
2023-01-01   0.010
2023-02-01  -0.011
2023-03-01   0.003
2023-04-01   0.010
2023-05-01   0.002
2023-06-01   0.010
2023-07-01   0.023
2023-08-01   0.012
2023-09-01   0.018
2023-10-01   0.010
2023-11-01   0.030
2023-12-01   0.022
2024-01-01  -0.009
2024-02-01   0.033
2024-03-01   0.020
2024-04-01  -0.015
2024-05-01   0.005
2024-06-01   0.038
2024-07-01   0.028
2024-08-01   0.009
2024-09-01   0.007
2024-10-01   0.010
2024-11-01   0.009
2024-12-01  -0.018
2025-01-01   0.013
2025-02-01   0.025
2025-03-01  -0.019
2025-04-01   0.008
2025-05-01   0.012
2025-06-01   0.014
2025-07-01   0.002
2025-08-01   0.017
2025-09-01   0.041
2025-10-01   0.026
2025-11-01  -0.009
2025-12-01  

In [24]:
df_ts_clean.to_csv('./hf_returns/tidan.csv')

In [25]:
import pandas as pd

# Extracting data from the third image
# Columns: JAN, FEB, MAR, APR, MAY, JUN, JUL, AUG, SEP, OCT, NOV, DEC, YEAR
data_three = {
    2020: [None, None, -2.00, 2.72, 0.04, 1.92, -0.85, -1.27, 0.14, 0.18, 1.97, 1.04, 3.86],
    2021: [-0.42, -5.08, 1.68, 3.22, 6.30, 2.73, 3.15, 2.97, 5.21, 0.65, 2.91, 5.34, 32.08],
    2022: [5.73, 3.88, 0.63, 1.70, 0.45, 1.63, 2.99, 4.92, -5.26, 5.06, 0.26, 1.05, 25.05],
    2023: [1.77, 0.39, 0.61, 0.24, 0.64, 1.45, -0.57, -0.54, -0.61, -0.45, 1.34, -0.08, 4.23],
    2024: [1.51, -1.23, -0.08, 0.47, 0.80, 1.00, -0.54, -0.10, -1.65, 1.72, 1.30, 0.89, 4.12],
    2025: [0.98, 1.75, 2.66, 2.78, 1.37, -1.72, 1.13, 1.40, 0.08, -0.97, -0.71, 1.11, 10.21],
    2026: [0.36, 1.12, -6.15, None, None, None, None, None, None, None, None, None, -4.76]
}

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'YEAR']
df_three = pd.DataFrame(data_three, index=months)

# 1. Exclude the annual 'YEAR' aggregate row
df_months = df_three.loc[df_three.index != 'YEAR']

# 2. Reshape/Unstack from matrix format to long format
df_long = df_months.unstack().reset_index()
df_long.columns = ['Year', 'Month', 'Return']

# 3. Create numerical month mappings and construct full Datetime index
month_map = {m: i+1 for i, m in enumerate(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])}
df_long['Month_Num'] = df_long['Month'].map(month_map)
df_long['Date'] = pd.to_datetime(df_long['Year'].astype(str) + '-' + df_long['Month_Num'].astype(str) + '-01')

# Sort chronologically and keep relevant columns
df_ts = df_long.sort_values('Date').reset_index(drop=True)[['Date', 'Return']]

# 4. Filter out leading and trailing missing data to isolate the true timeline
first_valid = df_ts['Return'].first_valid_index()
last_valid = df_ts['Return'].last_valid_index()
df_ts_clean = df_ts.loc[first_valid:last_valid].copy().reset_index(drop=True)

# Set Date as the formal index for your time-series analysis
df_ts_clean.set_index('Date', inplace=True)

# Save to a new CSV file
df_ts_clean = df_ts_clean /100
# df_ts_clean.to_csv('monthly_returns_timeseries_3.csv')

print(df_ts_clean)

            Return
Date              
2020-03-01 -0.0200
2020-04-01  0.0272
2020-05-01  0.0004
2020-06-01  0.0192
2020-07-01 -0.0085
...            ...
2025-11-01 -0.0071
2025-12-01  0.0111
2026-01-01  0.0036
2026-02-01  0.0112
2026-03-01 -0.0615

[73 rows x 1 columns]


In [26]:
df_ts_clean.to_csv('./hf_returns/maple_cap.csv')

In [27]:
import pandas as pd

# Extracting data from the fourth image (AGSF LP Fund)
# Columns: Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec, YTD
data_four = {
    2013: [None, None, None, 1.13, 1.63, 2.13, 1.24, 1.79, 1.82, 2.38, 1.15, 2.79, 17.24],
    2014: [1.49, 2.28, 1.89, 1.12, 1.30, 1.08, -0.27, 2.28, 1.91, 0.13, 2.36, 2.67, 19.79],
    2015: [1.52, 1.43, 1.62, 1.31, 1.99, 1.86, 1.44, -8.79, 4.28, -8.97, 1.62, 2.00, 0.26],
    2016: [1.24, 1.58, 1.75, 1.55, 1.21, 1.85, 0.95, 0.84, 1.06, 0.83, 0.91, 0.76, 15.53],
    2017: [1.07, 1.05, 1.00, 0.99, 1.38, 0.83, 0.63, 1.09, 0.89, 1.24, 0.08, 1.03, 11.88],
    2018: [1.34, 3.17, 1.54, 1.28, 0.20, 0.64, 1.08, 1.18, 1.02, 1.14, 1.01, 1.22, 15.84],
    2019: [0.75, 0.76, 0.70, 0.78, 1.00, 0.32, 0.93, 0.57, 1.04, 0.98, 0.70, 1.14, 10.11],
    2020: [0.01, 1.19, 1.26, 1.15, 0.85, 0.47, 0.93, 0.82, 1.23, 1.07, -3.14, 0.10, 6.02],
    2021: [0.93, 0.61, 0.76, 0.43, 0.96, 0.67, 0.52, 0.62, 0.64, 0.63, 0.15, 1.26, 8.49],
    2022: [1.34, 0.63, 0.78, 0.53, 0.70, 0.69, 0.68, 0.84, 1.33, 0.29, -7.05, -6.39, -5.95],
    2023: [1.22, 0.42, 0.28, 0.68, 1.62, -0.15, 1.55, 0.89, 1.59, 2.15, 2.15, -0.08, 12.98],
    2024: [1.85, 1.48, -0.26, 0.75, 1.10, 1.45, 2.40, 1.09, 2.64, 3.32, 2.72, 1.22, 21.59],
    2025: [0.64, 2.67, 0.59, 1.30, 0.68, 0.30, 0.34, 0.52, 1.10, -0.59, 0.00, 1.72, 9.63],
    2026: [0.08, 0.25, 2.08, 1.16, None, None, None, None, None, None, None, None, 3.61]
}

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'YTD']
df_four = pd.DataFrame(data_four, index=months)

# 1. Exclude the annual 'YTD' aggregate row
df_months = df_four.loc[df_four.index != 'YTD']

# 2. Reshape/Unstack from matrix format to long format
df_long = df_months.unstack().reset_index()
df_long.columns = ['Year', 'Month', 'Return']

# 3. Create numerical month mappings and construct full Datetime index
month_map = {m: i+1 for i, m in enumerate(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])}
df_long['Month_Num'] = df_long['Month'].map(month_map)
df_long['Date'] = pd.to_datetime(df_long['Year'].astype(str) + '-' + df_long['Month_Num'].astype(str) + '-01')

# Sort chronologically and keep relevant columns
df_ts = df_long.sort_values('Date').reset_index(drop=True)[['Date', 'Return']]

# 4. Filter out leading and trailing missing data to isolate the true timeline
first_valid = df_ts['Return'].first_valid_index()
last_valid = df_ts['Return'].last_valid_index()
df_ts_clean = df_ts.loc[first_valid:last_valid].copy().reset_index(drop=True)

# Set Date as the formal index for your time-series analysis
df_ts_clean.set_index('Date', inplace=True)

# Save to a new CSV file
df_ts_clean = df_ts_clean /100

print(df_ts_clean)

            Return
Date              
2013-04-01  0.0113
2013-05-01  0.0163
2013-06-01  0.0213
2013-07-01  0.0124
2013-08-01  0.0179
...            ...
2025-12-01  0.0172
2026-01-01  0.0008
2026-02-01  0.0025
2026-03-01  0.0208
2026-04-01  0.0116

[157 rows x 1 columns]


In [28]:
df_ts_clean.to_csv('./hf_returns/global_sigma.csv')

In [29]:
import pandas as pd

# Extracting only the EADF-B (net) rows from the screenshot
# Columns: Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec, Year
data_five = {
    2016: [None, None, None, None, None, None, None, None, None, 1.23, -0.64, -0.14, 0.44],
    2017: [0.59, 0.84, 1.73, 2.62, 2.42, 1.19, 0.50, 1.41, 0.86, 0.93, 0.72, 1.37, 16.25],
    2018: [0.73, 0.83, 1.34, 1.50, -0.20, -0.21, 1.46, 0.82, -0.19, 0.60, 0.56, 1.01, 8.54],
    2019: [2.46, 0.97, 2.43, 2.54, 1.86, 1.58, 3.03, 1.49, 2.02, 1.60, 1.09, 1.55, 25.07],
    2020: [2.37, 2.85, 4.30, 3.31, 3.36, 2.05, 0.36, 1.19, -1.10, 1.77, 0.85, 1.53, 25.24],
    2021: [0.69, 1.25, -0.51, 0.37, 0.55, 1.07, 1.53, 2.32, 1.41, -0.06, 0.13, 0.32, 9.42],
    2022: [-0.92, -2.97, 0.25, -5.22, -1.90, -14.07, 2.72, 5.88, -3.57, 7.07, 12.78, 3.60, 1.03],
    2023: [0.26, -0.56, -3.38, -6.56, 10.33, 7.81, 3.38, -1.16, -0.72, 3.76, 1.50, 1.75, 16.37],
    2024: [-1.09, 3.66, 9.42, -1.43, 2.45, 1.88, -1.82, 2.16, 0.13, 3.58, 0.96, 5.00, 27.30],
    2025: [4.60, 3.33, 0.58, -0.42, 1.38, 2.78, 3.55, 1.29, 6.02, 3.77, 1.36, 2.84, 35.66],
    2026: [5.69, 4.26, -4.98, 1.49, None, None, None, None, None, None, None, None, 6.28]
}

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Year']
df_five = pd.DataFrame(data_five, index=months)

# 1. Exclude the annual 'Year' aggregate row
df_months = df_five.loc[df_five.index != 'Year']

# 2. Reshape/Unstack from matrix format to long format
df_long = df_months.unstack().reset_index()
df_long.columns = ['Year', 'Month', 'Return']

# 3. Create numerical month mappings and construct full Datetime index
month_map = {m: i+1 for i, m in enumerate(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])}
df_long['Month_Num'] = df_long['Month'].map(month_map)
df_long['Date'] = pd.to_datetime(df_long['Year'].astype(str) + '-' + df_long['Month_Num'].astype(str) + '-01')

# Sort chronologically and keep relevant columns
df_ts = df_long.sort_values('Date').reset_index(drop=True)[['Date', 'Return']]

# 4. Filter out leading and trailing missing data to isolate the true timeline
first_valid = df_ts['Return'].first_valid_index()
last_valid = df_ts['Return'].last_valid_index()
df_ts_clean = df_ts.loc[first_valid:last_valid].copy().reset_index(drop=True)

# Set Date as the formal index for your time-series analysis
df_ts_clean.set_index('Date', inplace=True)

# Save to a new CSV file
# df_ts_clean.to_csv('eadf_b_returns_timeseries.csv')
df_ts_clean = df_ts_clean /100

print(df_ts_clean)

            Return
Date              
2016-10-01  0.0123
2016-11-01 -0.0064
2016-12-01 -0.0014
2017-01-01  0.0059
2017-02-01  0.0084
...            ...
2025-12-01  0.0284
2026-01-01  0.0569
2026-02-01  0.0426
2026-03-01 -0.0498
2026-04-01  0.0149

[115 rows x 1 columns]


In [30]:
df_ts_clean.to_csv('./hf_returns/enko.csv')

In [3]:
import io
import pandas as pd

# Cleaned data string
csv_data = """Month,Return
2008-01-31,1.83
2008-02-29,7.54
2008-03-31,-4.65
2008-04-30,5.95
2008-05-31,5.09
2008-06-30,4.91
2008-07-31,-2.87
2008-08-31,-0.43
2008-09-30,1.16
2008-10-31,1.20
2008-11-30,9.40
2008-12-31,3.69
2009-01-31,4.49
2009-02-28,4.09
2009-03-31,3.01
2009-04-30,0.76
2009-05-31,12.18
2009-06-30,-0.24
2009-07-31,3.68
2009-08-31,3.85
2009-09-30,4.51
2009-10-31,0.04
2009-11-30,8.66
2009-12-31,2.95
2010-01-31,1.66
2010-02-28,2.85
2010-03-31,5.34
2010-04-30,0.18
2010-05-31,3.47
2010-06-30,4.02
2010-07-31,5.45
2010-08-31,6.78
2010-09-30,4.87
2010-10-31,4.89
2010-11-30,0.57
2010-12-31,8.42
2011-01-31,10.49
2011-02-28,1.28
2011-03-31,2.61
2011-04-30,2.82
2011-10-31,3.92
2011-11-30,7.37
2011-12-31,2.56
2012-01-31,4.07
2012-02-29,2.02
2012-03-31,1.65
2012-04-30,1.34
2012-05-31,0.30
2012-06-30,-3.89
2012-07-31,5.73
2012-08-31,2.41
2012-09-30,-0.42
2012-10-31,0.53
2012-11-30,-1.23
2012-12-31,0.57
2013-01-31,2.99
2013-02-28,1.13
2013-03-31,0.48
2013-04-30,-2.33
2013-05-31,-0.32
2013-06-30,0.73
2013-07-31,0.45
2013-08-31,-5.05
2013-09-30,1.79
2013-10-31,0.30
2013-11-30,2.16
2013-12-31,3.70
2014-01-31,3.73
2014-02-28,-4.69
2014-03-31,-2.28
2014-04-30,3.35
2014-05-31,0.97
2014-06-30,0.01
2014-07-31,2.03
2014-08-31,4.09
2014-09-30,-0.02
2014-10-31,-3.10
2014-11-30,-0.12
2014-12-31,3.42
2015-01-31,1.11
2015-02-28,0.50
2015-03-31,0.65
2015-04-30,1.72
2015-05-31,2.52
2015-06-30,-0.19
2015-07-31,-1.67
2015-08-31,-3.86
2015-09-30,1.57
2015-10-31,-0.07
2015-11-30,0.86
2015-12-31,6.43
2016-01-31,3.16
2016-02-29,3.03
2016-03-31,2.93
2016-04-30,-4.11
2016-05-31,3.11
2016-06-30,-3.24
2016-07-31,-1.14
2016-08-31,3.34
2016-09-30,-1.09
2016-10-31,4.06
2016-11-30,1.23
2016-12-31,1.41
2017-01-31,2.78
2017-02-28,2.57
2017-03-31,1.20
2017-04-30,-0.49
2017-05-31,-1.14
2017-06-30,1.21
2017-07-31,-0.07
2017-08-31,1.79
2017-09-30,-0.58
2017-10-31,3.18
2017-11-30,4.54
2017-12-31,1.71
2018-01-31,2.60
2018-02-28,1.18
2018-03-31,-1.06
2018-04-30,2.11
2018-05-31,0.97
2018-06-30,-1.60
2018-07-31,5.77
2018-08-31,0.37
2018-09-30,-0.94
2018-10-31,-0.19
2018-11-30,0.32
2018-12-31,8.66
2019-01-31,2.10
2019-02-28,3.21
2019-03-31,2.36
2019-04-30,4.67
2019-05-31,-4.12
2019-06-30,2.21
2019-07-31,2.14
2019-08-31,-0.23
2019-09-30,-0.90
2019-10-31,5.89
2019-11-30,3.17
2019-12-31,3.74
2020-01-31,-1.12
2020-02-29,-3.89
2020-03-31,-8.06
2020-04-30,9.29
2020-05-31,6.77
2020-06-30,5.19
2020-07-31,2.13
2020-08-31,0.26
2020-09-30,0.55
2020-10-31,9.51
2020-11-30,4.67
2020-12-31,5.21
2021-01-31,4.22
2021-02-28,0.74
2021-03-31,3.25
2021-04-30,5.76
2021-05-31,3.22
2021-06-30,5.39
2021-07-31,1.89
2021-08-31,7.40
2021-09-30,2.32
2021-10-31,3.29
2021-11-30,4.12
2021-12-31,6.21
2022-01-31,6.12
2022-02-28,-0.12
2022-03-31,1.58
2022-04-30,7.76
2022-05-31,8.06
2022-06-30,-0.37
2022-07-31,0.30
2022-08-31,5.58
2022-09-30,2.00
2022-10-31,6.19
2022-11-30,3.72
2022-12-31,3.65
2023-01-31,5.52
2023-02-28,6.69
2023-03-31,0.06
2023-04-30,1.49
2023-05-31,5.36
2023-06-30,0.22
2023-07-31,3.55
2023-08-31,6.06
2023-09-30,-0.59
2023-10-31,4.02
2023-11-30,2.04
2023-12-31,1.37
2024-01-31,7.68
2024-02-29,4.64
2024-03-31,1.65
2024-04-30,-2.03
2024-05-31,3.82
2024-06-30,3.03
2024-07-31,1.04
2024-08-31,1.82
2024-09-30,-6.59
2024-10-31,9.39
2024-11-30,7.63
2024-12-31,5.76
2025-01-31,-1.58
2025-02-28,2.58
2025-03-31,5.04
2025-04-30,-10.51
2025-05-31,6.72
2025-06-30,3.51
2025-07-31,4.65
2025-08-31,5.48
2025-09-30,2.67
2025-10-31,3.55
2025-11-30,4.22
2025-12-31,1.43
2026-01-31,-4.94
2026-02-27,4.55
2026-03-31,-2.82
2026-04-30,8.34"""

# Parse the data directly into a DataFrame
df = pd.read_csv(io.StringIO(csv_data))

# Convert Month column to datetime objects
df["Month"] = pd.to_datetime(df["Month"])

# Apply your net return formula: (Value * 0.9) - (1% / 12)
df["Net_Return"] = (df["Return"] * 0.9) - (0.01 / 12)

print(df.head())

       Month  Return  Net_Return
0 2008-01-31    1.83    1.646167
1 2008-02-29    7.54    6.785167
2 2008-03-31   -4.65   -4.185833
3 2008-04-30    5.95    5.354167
4 2008-05-31    5.09    4.580167


In [8]:
(df.set_index('Month')/100)['Net_Return'].to_csv('./hf_returns/quanstream.csv')

In [9]:
import yfinance as yf

# Download historical data
tickers = ['^SP500TR']
data = yf.download(tickers, start='2000-01-01', interval='1mo')

sp500 = data['Close']

sp500.columns = ['SP500TR']

returns = sp500.pct_change().dropna()

returns

/var/folders/mc/qf75k40s6ns_nr8c35wdmp400000gn/T/ipykernel_13094/1224205126.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start='2000-01-01', interval='1mo')
[*********************100%***********************]  1 of 1 completed


,SP500TR
Date,
2000-02-01,-0.018929
2000-03-01,0.097829
2000-04-01,-0.030086
2000-05-01,-0.020518
2000-06-01,0.024654
...,...
2026-01-01,0.014500
2026-02-01,-0.007600
2026-03-01,-0.049795


In [11]:
returns.to_csv('./hf_returns/sp500.csv')

In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from the image
data = {
    "Year": [2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017, 2016, 2015],
    "Jan": [0.67, 1.06, -0.94, 0.25, -0.53, -0.22, 0.41, -0.01, -0.37, 1.25, None],
    "Feb": [1.72, -1.21, 1.12, 1.76, -0.17, 0.35, 2.07, -0.28, 0.13, 2.15, -0.11],
    "Mar": [1.96, 2.44, 3.43, 3.39, 0.20, 0.15, 4.10, 1.91, -0.34, -0.02, 5.36],
    "Apr": [1.50, -2.51, 4.65, -0.04, -0.21, 1.60, 3.78, -1.19, 0.70, 0.37, -1.07],
    "May": [1.31, 1.72, 1.61, 2.72, 0.58, 0.07, 0.95, 1.24, 0.27, -0.13, 3.37],
    "Jun": [-0.02, 0.62, 2.59, 2.22, 2.80, 1.26, 1.30, 1.78, 0.35, 0.87, 9.08],
    "Jul": [0.67, 1.40, -0.88, 2.21, 3.06, 2.47, 1.16, 1.27, 1.27, 0.59, 3.93],
    "Aug": [-0.69, -0.16, 1.31, -2.42, 0.15, 3.61, 1.04, 0.71, 2.30, 0.14, 2.21],
    "Sep": [None, -2.87, 0.48, -1.29, 2.24, -0.16, 1.15, 0.60, 1.95, -0.59, -0.25],
    "Oct": [None, 3.88, 0.68, -0.73, -1.39, 3.09, 0.31, 1.86, 0.94, 0.07, 0.02],
    "Nov": [None, 3.13, 2.16, 0.30, 1.02, -0.98, -2.29, 1.32, 0.20, 0.14, 1.04],
    "Dec": [None, 0.27, -0.85, 1.79, -1.90, 1.09, -1.10, 0.08, -0.29, -0.69, 0.39]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
# Convert month abbreviations to numeric strings (e.g., 'Jan' -> '01')
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Drops non-existent months (Jan 2015 and Sep-Dec 2025)
)

# Convert return values from percentages to floating fractions (e.g., 0.67 -> 0.0067)
df_final["Return"] = df_final["Return"] / 100

# Save output to file
df_final

,Return
Period,
2015-02,-0.0011
2015-03,0.0536
2015-04,-0.0107
2015-05,0.0337
2015-06,0.0908
...,...
2025-04,0.0150
2025-05,0.0131
2025-06,-0.0002


In [2]:
df_final.to_csv('./hf_returns/wizard_quant.csv')

In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from the image
data = {
    "Year": [2025, 2024, 2023, 2022, 2021, 2020, 2019],
    "Jan": [2.2, 2.5, -2.5, 5.6, 2.0, 3.3, None],
    "Feb": [-2.3, 5.9, 1.1, 1.3, -0.5, -3.9, None],
    "Mar": [-1.9, 0.9, 4.8, -1.8, 2.2, -1.2, 2.9],
    "Apr": [1.6, -0.7, 2.0, 6.5, 8.0, 10.4, 5.1],
    "May": [3.8, 2.4, 2.4, 2.6, -2.8, 16.5, -1.6],
    "Jun": [2.3, 3.9, 0.9, 4.2, 4.2, 10.3, 7.2],
    "Jul": [3.0, -2.1, 2.6, -2.8, 4.6, 12.5, 3.7],
    "Aug": [0.2, 1.7, 0.5, -1.0, 4.9, 4.3, -0.8],
    "Sep": [4.4, 4.0, 1.7, 4.7, 5.3, -0.6, 1.2],
    "Oct": [2.2, 1.5, 2.0, -0.2, 2.3, -0.7, 3.0],
    "Nov": [-0.3, 2.7, 1.0, -3.3, 4.3, 8.8, 4.5],
    "Dec": [0.6, -1.4, 5.3, 3.1, -0.4, 2.9, 1.2]
}

df_raw = pd.DataFrame(data)

# Note: The 'YTD' column from the image is omitted here since it is an aggregate 
# value and will be implicitly calculated if you compound these monthly returns.

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
# Convert month abbreviations to numeric strings (e.g., 'Jan' -> '01')
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Drops non-existent months (Jan and Feb 2019)
)

# Convert return values from percentages to floating fractions (e.g., 2.2 -> 0.022)
df_final["Return"] = df_final["Return"] / 100

# Save output to file
df_final

,Return
Period,
2019-03,0.029
2019-04,0.051
2019-05,-0.016
2019-06,0.072
2019-07,0.037
...,...
2025-08,0.002
2025-09,0.044
2025-10,0.022


In [3]:
df_final.to_csv('./hf_returns/coban_pod_1.csv')

In [4]:
import pandas as pd

# 1. Reconstruct the raw grid data from the new image
data = {
    "Year": [2025, 2024],
    "Jan": [0.4, 0.8],
    "Feb": [-2.0, 3.8],
    "Mar": [-2.3, -0.0],
    "Apr": [1.9, 0.4],
    "May": [1.8, 2.3],
    "Jun": [1.4, 4.4],
    "Jul": [1.2, -1.7],
    "Aug": [-0.5, 0.6],
    "Sep": [3.4, 0.6],
    "Oct": [1.9, 1.2],
    "Nov": [1.6, -0.3],
    "Dec": [-0.4, 5.1]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()
)

# Convert return values from percentages to floating fractions (e.g., 0.4 -> 0.004)
df_final["Return"] = df_final["Return"] / 100

# Save output to file
df_final

,Return
Period,
2024-01,0.008
2024-02,0.038
2024-03,-0.000
2024-04,0.004
2024-05,0.023
2024-06,0.044
2024-07,-0.017
2024-08,0.006
2024-09,0.006


In [5]:
df_final.to_csv('./hf_returns/coban_pod_2.csv')

In [6]:
import pandas as pd

# 1. Reconstruct the raw grid data from the third image
data = {
    "Year": [2025, 2024, 2023],
    "Jan": [2.0, 1.4, 0.0],
    "Feb": [-2.2, 6.3, -0.8],
    "Mar": [1.9, 2.0, -12.9],
    "Apr": [6.3, 1.2, 2.8],
    "May": [0.1, -1.5, 0.2],
    "Jun": [-0.4, 4.7, 2.2],
    "Jul": [2.0, 2.4, -0.9],
    "Aug": [-4.6, 0.9, 9.1],
    "Sep": [-1.2, 2.1, 6.1],
    "Oct": [-1.2, 0.2, 1.0],
    "Nov": [2.0, 3.2, 6.3],
    "Dec": [2.4, 3.9, 2.8]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()
)

# Convert return values from percentages to floating fractions (e.g., 2.0 -> 0.02)
df_final["Return"] = df_final["Return"] / 100

# Save output to file
df_final

,Return
Period,
2023-01,0.000
2023-02,-0.008
2023-03,-0.129
2023-04,0.028
2023-05,0.002
2023-06,0.022
2023-07,-0.009
2023-08,0.091
2023-09,0.061


In [7]:
df_final.to_csv('./hf_returns/coban_orca.csv')

In [8]:
import pandas as pd

# 1. Reconstruct the raw grid data from the fourth image
data = {
    "Year": [2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019],
    "Jan": [15.0, 4.5, 3.1, None, -7.6, -0.5, 0.6, 2.4],
    "Feb": [None, 0.4, 7.0, None, 1.1, 5.8, 2.8, 4.7],
    "Mar": [None, -7.6, 4.2, None, 5.1, 1.5, -10.2, 2.7],
    "Apr": [None, -1.4, -4.7, None, -4.4, 6.7, 6.7, 4.9],
    "May": [None, 6.4, 0.4, None, -0.2, 3.0, 7.3, -4.3],
    "Jun": [None, 6.7, 1.4, 2.7, -2.1, 5.8, 4.4, 3.9],
    "Jul": [None, -0.4, -0.1, 7.6, 2.8, 0.2, 3.0, 3.3],
    "Aug": [None, 5.3, -2.1, -2.3, 1.1, 1.9, 6.5, 3.7],
    "Sep": [None, 3.5, -0.9, -1.7, -4.1, -2.2, 0.9, 3.6],
    "Oct": [None, 8.6, 0.0, -4.5, -1.1, 9.7, 2.0, 2.0],
    "Nov": [None, -1.0, 3.0, 6.7, 9.9, -2.6, 15.7, -1.5],
    "Dec": [None, 0.7, -2.0, 3.6, -4.8, -0.3, 4.3, 5.1]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Smoothly filters out the empty months (e.g., Feb-Dec 2026, Jan-May 2023)
)

# Convert return values from percentages to floating fractions (e.g., 15.0 -> 0.15)
df_final["Return"] = df_final["Return"] / 100

# Save output to file
df_final

,Return
Period,
2019-01,0.024
2019-02,0.047
2019-03,0.027
2019-04,0.049
2019-05,-0.043
...,...
2025-09,0.035
2025-10,0.086
2025-11,-0.010


In [9]:
df_final.to_csv('./hf_returns/coban_amount_abs_ret.csv')

In [10]:
import pandas as pd

# 1. Reconstruct the raw grid data from the fifth image
data = {
    "Year": [2026, 2025, 2024, 2023, 2022, 2021],
    "Jan": [-0.6, -4.0, -2.2, 4.3, 3.8, None],
    "Feb": [2.2, 0.3, 6.3, 6.8, -3.1, 5.0],
    "Mar": [None, 2.9, 2.5, 6.5, 5.2, -0.2],
    "Apr": [None, 3.0, 2.6, 7.3, 8.8, -0.3],
    "May": [None, 1.3, -3.5, 0.9, 1.6, 6.9],
    "Jun": [None, 3.0, -3.4, -2.7, 5.9, 2.4],
    "Jul": [None, 0.6, 2.2, 0.2, -4.2, 4.0],
    "Aug": [None, -2.1, 3.4, 0.8, 6.4, -1.1],
    "Sep": [None, 1.9, 9.1, -1.3, 1.2, 6.4],
    "Oct": [None, 2.4, 3.2, 0.8, 7.5, -0.4],
    "Nov": [None, 1.8, -1.3, -0.0, -0.1, -1.6],
    "Dec": [None, 3.0, 2.9, 0.9, -4.5, 8.3]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out the unrated/future months (e.g., Mar-Dec 2026, Jan 2021)
)

# Convert return values from percentages to floating fractions (e.g., 2.2 -> 0.022)
df_final["Return"] = df_final["Return"] / 100

# Save output to file
df_final

,Return
Period,
2021-02,0.050
2021-03,-0.002
2021-04,-0.003
2021-05,0.069
2021-06,0.024
...,...
2025-10,0.024
2025-11,0.018
2025-12,0.030


In [11]:
df_final.to_csv('./hf_returns/coban_edge_capital.csv')

In [12]:
import pandas as pd

# 1. Reconstruct the raw grid data from the sixth image
data = {
    "Year": [2026, 2025, 2024, 2023],
    "Jan": [2.7, 1.8, 2.2, None],
    "Feb": [0.4, 0.8, 1.0, None],
    "Mar": [None, 1.2, 3.2, None],
    "Apr": [None, 1.3, 2.9, None],
    "May": [None, 1.9, 5.8, 1.8],
    "Jun": [None, 0.9, 3.5, 3.1],
    "Jul": [None, 3.5, 1.9, 2.4],
    "Aug": [None, -1.3, 2.3, 3.2],
    "Sep": [None, 0.6, 1.3, 4.2],
    "Oct": [None, 2.8, 3.1, 2.0],
    "Nov": [None, 0.9, 2.9, 3.5],
    "Dec": [None, 2.2, 0.9, 5.3]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out the unrated/future months (e.g., Mar-Dec 2026, Jan-Apr 2023)
)

# Convert return values from percentages to floating fractions (e.g., 2.7 -> 0.027)
df_final["Return"] = df_final["Return"] / 100

# Save output to file
df_final

,Return
Period,
2023-05,0.018
2023-06,0.031
2023-07,0.024
2023-08,0.032
2023-09,0.042
2023-10,0.020
2023-11,0.035
2023-12,0.053
2024-01,0.022


In [13]:
df_final.to_csv('./hf_returns/coban_arkis_ls_alpha.csv')

In [14]:
import pandas as pd

# 1. Reconstruct the raw grid data from the seventh image
data = {
    "Year": [2025, 2024, 2023, 2022, 2021],
    "Jan": [0.3, 1.3, 7.5, 2.4, 2.4],
    "Feb": [5.2, 0.9, 4.5, 1.4, 3.9],
    "Mar": [5.1, 6.4, 2.3, 6.2, 2.6],
    "Apr": [4.4, 5.1, 2.8, 4.9, 1.7],
    "May": [2.4, 3.8, 2.7, 0.3, 0.6],
    "Jun": [-1.1, 1.8, 3.2, 4.8, 1.7],
    "Jul": [1.9, 5.0, 0.6, 2.2, -0.4],
    "Aug": [-1.7, 3.8, 0.6, 3.4, -0.7],
    "Sep": [3.8, 4.5, -0.2, 3.0, 3.0],
    "Oct": [3.0, 6.4, 3.7, 3.2, 3.0],
    "Nov": [4.2, 0.7, 1.9, 6.7, 2.1],
    "Dec": [-4.9, -0.0, 2.9, 5.3, 0.8]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()
)

# Convert return values from percentages to floating fractions (e.g., 0.3 -> 0.003)
df_final["Return"] = df_final["Return"] / 100

# Save output to file
df_final

,Return
Period,
2021-01,0.024
2021-02,0.039
2021-03,0.026
2021-04,0.017
2021-05,0.006
2021-06,0.017
2021-07,-0.004
2021-08,-0.007
2021-09,0.030


In [15]:
df_final.to_csv('./hf_returns/coban_clarion_multi_asset.csv')

In [16]:
import pandas as pd

# 1. Reconstruct the raw grid data from the eighth image
data = {
    "Year": [2026, 2025, 2024, 2023],
    "Jan": [1.5, 0.9, 3.8, 5.7],
    "Feb": [3.7, -1.3, 5.0, 2.0],
    "Mar": [None, 0.8, 3.3, 11.4],
    "Apr": [None, 6.5, 0.1, 2.0],
    "May": [None, 1.7, 6.8, 1.9],
    "Jun": [None, 3.7, 5.7, 1.8],
    "Jul": [None, 0.2, 5.4, 2.4],
    "Aug": [None, -0.2, 13.0, 0.3],
    "Sep": [None, 3.1, 5.1, 0.8],
    "Oct": [None, 6.1, 1.8, 6.6],
    "Nov": [None, 0.5, 1.3, 8.1],
    "Dec": [None, -0.7, 2.9, 10.7]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out the unrated/future months (e.g., Mar-Dec 2026)
)

# Convert return values from percentages to floating fractions (e.g., 1.5 -> 0.015)
df_final["Return"] = df_final["Return"] / 100

# Save output to file
df_final

,Return
Period,
2023-01,0.057
2023-02,0.020
2023-03,0.114
2023-04,0.020
2023-05,0.019
2023-06,0.018
2023-07,0.024
2023-08,0.003
2023-09,0.008


In [17]:
df_final.to_csv('./hf_returns/coban_plettenberg.csv')

In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from the ninth image (Capstone Global)
data = {
    "Year": [
        2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017, 
        2016, 2015, 2014, 2013, 2012, 2011, 2010, 2009, 2008, 2007
    ],
    "Jan": [1.82, 2.79, 1.87, 2.26, 2.65, 0.10, 1.62, 1.88, 0.15, 1.59, -0.28, -0.17, 2.09, -1.90, 1.89, 3.66, 2.56, 1.98, 8.88, None],
    "Feb": [1.55, 1.60, 0.64, 1.52, -1.11, -1.17, 0.57, 0.25, 1.24, -0.17, -0.65, 1.46, 1.04, -0.05, 0.90, 1.16, 0.06, 1.23, 0.13, None],
    "Mar": [-3.80, 0.39, 1.09, -1.29, 0.52, 0.85, -4.62, 1.23, 1.42, 0.39, 1.26, 0.28, -0.64, 0.56, 1.30, -0.14, 1.55, 0.62, -0.74, None],
    "Apr": [1.42, -0.89, 0.54, 0.60, 1.52, 0.10, 3.65, 0.65, 1.64, 0.89, 1.75, 1.29, 1.11, 0.51, 0.34, 1.17, 1.80, 0.16, 0.71, None],
    "May": [None, 0.88, 0.61, 0.83, -1.10, 0.26, 0.05, 0.84, -5.34, 1.16, 0.73, 0.79, 0.61, 0.48, -1.37, -0.35, -1.68, 1.07, -0.33, None],
    "Jun": [None, 0.48, -0.53, -0.31, -1.66, -0.26, 0.97, 0.74, 0.84, 0.61, 1.68, -1.15, 0.00, -0.90, 1.39, 0.16, -0.13, 1.40, -2.80, None],
    "Jul": [None, 0.21, 1.72, 1.11, -0.39, -0.74, 0.93, 0.90, 0.02, 1.20, 2.01, 0.40, -0.77, 0.39, 2.52, 3.68, -0.47, 0.58, 1.56, None],
    "Aug": [None, 0.57, -1.30, 1.37, 0.76, 0.38, 0.69, 0.49, -0.15, 0.21, 0.18, 1.24, -1.98, 0.10, -1.73, 2.08, 0.09, -0.47, -0.87, 9.53],
    "Sep": [None, 1.60, -0.11, 0.94, -0.19, 1.37, 1.27, 0.37, 0.15, 0.05, 1.56, -1.28, -0.78, 0.00, 0.55, -2.40, 1.44, 1.76, 1.44, 2.37],
    "Oct": [None, 1.20, -1.07, -0.33, 0.42, -5.19, 0.16, 0.52, 1.01, 1.07, -0.81, 0.68, -1.31, 0.47, 0.82, 2.24, 1.54, 1.02, -11.28, 1.07],
    "Nov": [None, 1.21, 1.90, 0.53, -0.28, 1.21, 2.05, 1.07, 0.68, 0.45, -0.26, -0.79, -0.85, -0.45, 0.26, -1.16, 1.91, -0.40, -5.11, 4.42],
    "Dec": [None, 1.12, 1.03, 0.24, 2.15, 1.75, 1.47, 0.94, -0.92, -0.17, 1.57, 1.27, -1.12, 0.37, 0.31, 1.88, 0.82, 1.03, 3.86, 0.73]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out future months in 2026 and pre-inception months in 2007
)

# Convert return values from percentages to floating fractions (e.g., 1.82 -> 0.0182)
df_final["Return"] = df_final["Return"] / 100

# Save output to file
df_final

,Return
Period,
2007-08,0.0953
2007-09,0.0237
2007-10,0.0107
2007-11,0.0442
2007-12,0.0073
...,...
2025-12,0.0112
2026-01,0.0182
2026-02,0.0155


In [2]:
df_final.to_csv('./hf_returns/capstone.csv')

In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from the provided image
data = {
    "Year": [
        2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 
        2017, 2016, 2015, 2014, 2013, 2012, 2011, 2010, 2009, 2008
    ],
    "Jan": [5.48, 7.03, 1.51, 15.60, 1.07, 0.00, -2.26, 17.76, 8.59, 1.19, -12.35, -6.78, 0.27, 8.26, 4.55, 14.03, -2.97, 22.60, None],
    "Feb": [-6.28, -2.05, 3.54, 1.55, 0.68, 7.09, -11.02, 4.44, -2.36, 5.58, 2.02, 3.56, 8.12, 3.97, 1.65, 9.26, 6.01, 7.00, None],
    "Mar": [-6.15, -5.15, 5.34, -13.35, -5.70, 3.32, -36.23, -2.60, -4.57, -3.54, 8.77, -2.34, -0.48, 4.11, 7.51, -4.00, 4.55, 19.23, None],
    "Apr": [7.17, 0.42, -2.36, 7.00, -3.99, 3.20, 22.38, 4.25, 1.20, 1.09, 4.68, 3.67, -2.69, 3.80, -1.37, 1.20, 5.77, 11.00, None],
    "May": [None, 7.87, 4.32, -1.06, 1.45, 1.16, 7.75, -4.74, 0.44, -3.75, 3.00, 0.74, -0.49, 5.89, -0.67, 6.43, -3.00, 17.19, None],
    "Jun": [None, 5.70, -0.54, 6.74, -12.06, -1.95, 10.71, 4.58, -0.12, 3.02, -9.79, -0.90, 0.88, -3.78, 3.99, 1.32, -17.98, 20.93, None],
    "Jul": [None, 6.62, 11.56, 10.35, 8.13, -0.01, 5.46, 1.05, 4.06, 4.78, 12.80, -3.78, -2.27, 2.70, 1.94, 0.36, 3.93, 7.90, -1.89],
    "Aug": [None, 4.05, -1.62, -2.47, 3.44, 1.50, 7.01, -4.86, 0.22, -3.21, 4.95, -4.55, 1.44, -3.51, -1.57, -5.00, -6.65, 15.28, -7.24],
    "Sep": [None, 0.32, 0.97, -0.17, -13.40, 0.78, 0.29, 7.60, -1.31, 4.67, -0.77, -5.96, -1.87, -0.71, 2.40, -5.34, 7.03, -0.50, -21.90],
    "Oct": [None, -5.25, 1.63, -2.52, 10.84, 1.35, 6.24, 0.98, -7.37, -1.12, 1.72, 4.60, -2.89, 5.06, 7.61, 2.76, 7.73, -12.63, 16.63],
    "Nov": [None, 5.69, 18.19, 9.25, 3.93, -0.94, 15.61, 2.87, -0.29, 3.50, 23.95, 2.49, -0.04, 4.73, 1.72, -0.41, 5.61, -0.87, -7.93],
    "Dec": [None, 4.00, -4.97, 9.63, -4.04, 6.52, 5.17, 3.32, -14.01, 5.14, 5.67, -9.85, -0.52, 2.68, 3.01, -4.34, 5.13, 8.65, 11.02]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out the future months of 2026 and pre-launch months of 2008
)

# Convert return values from percentages to floating fractions (e.g., 5.48 -> 0.0548)
df_final["Return"] = df_final["Return"] / 100

# Save output to file or display
print(df_final)

         Return
Period         
2008-07 -0.0189
2008-08 -0.0724
2008-09 -0.2190
2008-10  0.1663
2008-11 -0.0793
...         ...
2025-12  0.0400
2026-01  0.0548
2026-02 -0.0628
2026-03 -0.0615
2026-04  0.0717

[214 rows x 1 columns]


In [2]:
df_final.to_csv('./hf_returns/gator.csv')

In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from the provided image
data = {
    "Year": [2026, 2025, 2024, 2023, 2022],
    "Jan": [12.32, -0.55, 7.19, 8.98, None],
    "Feb": [9.17, -2.38, 3.35, 2.03, None],
    "Mar": [-0.91, -0.28, 0.67, 1.64, None],
    "Apr": [13.35, -0.05, 1.07, -0.97, None],
    "May": [7.01, 2.25, 7.66, 11.03, None],
    "Jun": [None, 6.20, 5.36, 0.21, None],
    "Jul": [None, 6.35, 0.51, 2.39, None],
    "Aug": [None, 6.60, 0.68, 0.59, None],
    "Sep": [None, 6.52, 7.34, -2.62, None],
    "Oct": [None, 11.21, 1.92, -3.90, -4.73],
    "Nov": [None, 0.51, 6.18, 3.93, 3.49],
    "Dec": [None, 1.15, 3.48, 4.67, 0.09]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out the future months of 2026 and pre-launch months of 2022
)

# Convert return values from percentages to floating fractions (e.g., 12.32 -> 0.1232)
df_final["Return"] = df_final["Return"] / 100

# Save output to file or display
print(df_final)

         Return
Period         
2022-10 -0.0473
2022-11  0.0349
2022-12  0.0009
2023-01  0.0898
2023-02  0.0203
2023-03  0.0164
2023-04 -0.0097
2023-05  0.1103
2023-06  0.0021
2023-07  0.0239
2023-08  0.0059
2023-09 -0.0262
2023-10 -0.0390
2023-11  0.0393
2023-12  0.0467
2024-01  0.0719
2024-02  0.0335
2024-03  0.0067
2024-04  0.0107
2024-05  0.0766
2024-06  0.0536
2024-07  0.0051
2024-08  0.0068
2024-09  0.0734
2024-10  0.0192
2024-11  0.0618
2024-12  0.0348
2025-01 -0.0055
2025-02 -0.0238
2025-03 -0.0028
2025-04 -0.0005
2025-05  0.0225
2025-06  0.0620
2025-07  0.0635
2025-08  0.0660
2025-09  0.0652
2025-10  0.1121
2025-11  0.0051
2025-12  0.0115
2026-01  0.1232
2026-02  0.0917
2026-03 -0.0091
2026-04  0.1335
2026-05  0.0701


In [2]:
df_final.to_csv('./hf_returns/singularity_mf.csv')

In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from the provided image
data = {
    "Year": [
        2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 
        2017, 2016, 2015, 2014
    ],
    "Jan": [1.37, 0.86, 1.99, 0.78, -1.33, 1.63, 1.40, 1.40, 3.88, 1.63, -0.53, 0.81, None],
    "Feb": [0.96, 1.09, 0.20, 0.04, 1.04, 3.47, 0.83, 2.06, -0.10, -0.50, 0.34, 1.54, None],
    "Mar": [-1.96, -1.46, 0.71, 0.52, -0.66, -1.22, -0.40, 0.80, 0.71, 1.36, -1.21, 1.46, None],
    "Apr": [3.22, 0.79, 0.35, 0.72, 0.65, 1.54, 2.28, 1.39, 0.96, 0.66, 0.56, 0.42, None],
    "May": [0.70, 2.23, 0.19, 0.53, 0.49, 0.77, 2.14, 0.23, 1.20, -0.56, 1.80, 1.75, None],
    "Jun": [None, 2.26, 0.03, 0.06, -0.40, 0.66, 2.62, 0.47, 0.63, 0.81, -0.15, 0.06, None],
    "Jul": [None, 2.37, 0.01, 0.27, -1.15, -0.30, 1.31, 0.28, -0.39, 0.89, 1.31, 0.52, -0.28],
    "Aug": [None, 1.42, -0.07, 1.17, 0.08, 1.11, 0.92, 0.11, 0.51, 3.46, 0.39, 0.75, 0.28],
    "Sep": [None, 2.75, 0.21, 1.32, -0.28, 1.76, 0.85, -0.85, 0.92, 1.75, 1.27, 0.67, -0.34],
    "Oct": [None, 2.48, 1.04, -1.19, 0.66, 1.50, 0.44, 1.17, -1.18, 1.96, -0.14, 0.34, -0.49],
    "Nov": [None, 0.45, 1.29, 1.01, 0.45, 0.22, 4.74, 1.56, -1.24, 0.47, -0.05, 0.89, 1.00],
    "Dec": [None, 1.62, 0.88, 1.31, 1.76, -0.23, 5.76, 2.65, 0.10, 3.10, 0.23, 0.50, 1.34]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out future months of 2026 and pre-launch months of 2014
)

# Convert return values from percentages to floating fractions (e.g., 1.37 -> 0.0137)
df_final["Return"] = df_final["Return"] / 100

# Save output to file or display
print(df_final)

         Return
Period         
2014-07 -0.0028
2014-08  0.0028
2014-09 -0.0034
2014-10 -0.0049
2014-11  0.0100
...         ...
2026-01  0.0137
2026-02  0.0096
2026-03 -0.0196
2026-04  0.0322
2026-05  0.0070

[143 rows x 1 columns]


In [2]:
df_final.to_csv('./hf_returns/boothbay.csv')

In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from the provided image
data = {
    "Year": [2026, 2025, 2024, 2023, 2022, 2021],
    "Jan": [23.5, 9.0, 11.3, 15.9, -11.6, None],
    "Feb": [17.1, -6.0, 27.3, -14.6, -10.2, None],
    "Mar": [-8.0, -3.0, 21.2, 9.7, 9.6, None],
    "Apr": [9.1, -27.4, -14.0, 4.8, 6.2, None],
    "May": [None, 5.8, -8.5, -14.7, 12.8, None],
    "Jun": [None, 12.5, 12.8, 25.7, -10.4, None],
    "Jul": [None, 16.7, 1.9, 8.0, 19.1, None],
    "Aug": [None, -6.3, -2.8, -4.3, -10.7, 1.7],
    "Sep": [None, 23.6, -3.8, -6.9, -22.0, -6.4],
    "Oct": [None, 9.1, -6.7, -22.6, 22.5, -5.1],
    "Nov": [None, -4.0, 16.3, 27.7, -8.7, -19.3],
    "Dec": [None, 4.9, -6.5, 6.2, -18.3, 11.0]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out future months of 2026 and pre-launch months of 2021
)

# Convert return values from percentages to floating fractions (e.g., 23.5 -> 0.235)
df_final["Return"] = df_final["Return"] / 100

# Save output to file or display
print(df_final)

         Return
Period         
2021-08   0.017
2021-09  -0.064
2021-10  -0.051
2021-11  -0.193
2021-12   0.110
2022-01  -0.116
2022-02  -0.102
2022-03   0.096
2022-04   0.062
2022-05   0.128
2022-06  -0.104
2022-07   0.191
2022-08  -0.107
2022-09  -0.220
2022-10   0.225
2022-11  -0.087
2022-12  -0.183
2023-01   0.159
2023-02  -0.146
2023-03   0.097
2023-04   0.048
2023-05  -0.147
2023-06   0.257
2023-07   0.080
2023-08  -0.043
2023-09  -0.069
2023-10  -0.226
2023-11   0.277
2023-12   0.062
2024-01   0.113
2024-02   0.273
2024-03   0.212
2024-04  -0.140
2024-05  -0.085
2024-06   0.128
2024-07   0.019
2024-08  -0.028
2024-09  -0.038
2024-10  -0.067
2024-11   0.163
2024-12  -0.065
2025-01   0.090
2025-02  -0.060
2025-03  -0.030
2025-04  -0.274
2025-05   0.058
2025-06   0.125
2025-07   0.167
2025-08  -0.063
2025-09   0.236
2025-10   0.091
2025-11  -0.040
2025-12   0.049
2026-01   0.235
2026-02   0.171
2026-03  -0.080
2026-04   0.091


In [2]:
df_final.to_csv('./hf_returns/quantos.csv')

In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from the provided image (CKMF B-2 Net Return only)
data = {
    "Year": [2026, 2025, 2024, 2023, 2022, 2021, 2020],
    "Jan": [6.2, 1.7, 1.0, 6.6, 0.1, 5.4, None],
    "Feb": [4.4, -2.4, 5.0, 2.1, 0.6, 14.0, None],
    "Mar": [-5.6, -15.4, 1.3, -6.8, 6.9, 10.6, None],
    "Apr": [21.2, -3.4, -4.8, -3.2, -0.5, 6.8, None],
    "May": [None, 14.4, 7.9, -4.7, 0.5, 4.1, None],
    "Jun": [None, 7.9, 1.5, 10.7, -10.2, 3.6, None],
    "Jul": [None, 3.8, 3.6, 6.0, 4.7, -5.8, None],
    "Aug": [None, -0.9, -1.5, -1.3, 6.4, 5.4, None],
    "Sep": [None, 4.7, 8.5, -1.6, -7.4, 2.4, None],
    "Oct": [None, 3.4, -0.2, -8.2, 5.1, 8.9, -3.2],
    "Nov": [None, -6.2, 4.7, 7.9, -0.2, -4.3, 14.1],
    "Dec": [None, 0.1, -1.9, 10.3, -6.2, 0.9, 9.7]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out future months of 2026 and pre-launch months of 2020
)

# Convert return values from percentages to floating fractions (e.g., 6.2 -> 0.062)
df_final["Return"] = df_final["Return"] / 100

# Save output to file or display
print(df_final)

         Return
Period         
2020-10  -0.032
2020-11   0.141
2020-12   0.097
2021-01   0.054
2021-02   0.140
...         ...
2025-12   0.001
2026-01   0.062
2026-02   0.044
2026-03  -0.056
2026-04   0.212

[67 rows x 1 columns]


In [2]:
df_final.to_csv('./hf_returns/castleknight.csv')

In [3]:
import pandas as pd

# 1. Reconstruct the raw grid data from the provided image
data = {
    "Year": [2026, 2025, 2024],
    "Jan": [0.8, 5.5, 3.5],
    "Feb": [-0.3, 0.4, 1.8],
    "Mar": [7.8, -3.8, -1.2],
    "Apr": [-1.1, 0.7, -0.3],
    "May": [-0.4, 3.7, -1.1],
    "Jun": [None, 0.8, 5.2],
    "Jul": [None, 0.2, -2.2],
    "Aug": [None, -0.8, -2.0],
    "Sep": [None, 3.2, 3.2],
    "Oct": [None, 1.5, 1.1],
    "Nov": [None, -2.0, 5.0],
    "Dec": [None, 0.3, 5.2]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out future months of 2026
)

# Convert return values from percentages to floating fractions (e.g., 0.8 -> 0.008)
df_final["Return"] = df_final["Return"] / 100

# Save output to file or display
print(df_final)

         Return
Period         
2024-01   0.035
2024-02   0.018
2024-03  -0.012
2024-04  -0.003
2024-05  -0.011
2024-06   0.052
2024-07  -0.022
2024-08  -0.020
2024-09   0.032
2024-10   0.011
2024-11   0.050
2024-12   0.052
2025-01   0.055
2025-02   0.004
2025-03  -0.038
2025-04   0.007
2025-05   0.037
2025-06   0.008
2025-07   0.002
2025-08  -0.008
2025-09   0.032
2025-10   0.015
2025-11  -0.020
2025-12   0.003
2026-01   0.008
2026-02  -0.003
2026-03   0.078
2026-04  -0.011
2026-05  -0.004


In [4]:
df_final.to_csv('./hf_returns/cfm_x15.csv')

In [5]:
import pandas as pd

# 1. Reconstruct the raw grid data from the provided image
data = {
    "Year": [2026, 2025, 2024],
    "Jan": [2.00, 12.54, -5.61],
    "Feb": [-8.84, 8.77, 11.57],
    "Mar": [-19.02, -8.33, 3.54],
    "Apr": [11.86, -1.66, -4.64],
    "May": [None, 7.50, 8.09],
    "Jun": [None, 5.25, 7.52],
    "Jul": [None, -3.07, -1.05],
    "Aug": [None, 6.81, 3.08],
    "Sep": [None, 10.12, 21.00],
    "Oct": [None, -2.56, -6.86],
    "Nov": [None, -2.43, 8.42],
    "Dec": [None, -0.93, -0.06]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out future months of 2026
)

# Convert return values from percentages to floating fractions (e.g., 2.00 -> 0.02)
df_final["Return"] = df_final["Return"] / 100

# Save output to file or display
print(df_final)

         Return
Period         
2024-01 -0.0561
2024-02  0.1157
2024-03  0.0354
2024-04 -0.0464
2024-05  0.0809
2024-06  0.0752
2024-07 -0.0105
2024-08  0.0308
2024-09  0.2100
2024-10 -0.0686
2024-11  0.0842
2024-12 -0.0006
2025-01  0.1254
2025-02  0.0877
2025-03 -0.0833
2025-04 -0.0166
2025-05  0.0750
2025-06  0.0525
2025-07 -0.0307
2025-08  0.0681
2025-09  0.1012
2025-10 -0.0256
2025-11 -0.0243
2025-12 -0.0093
2026-01  0.0200
2026-02 -0.0884
2026-03 -0.1902
2026-04  0.1186


In [6]:
df_final.to_csv('./hf_returns/orgeuil.csv')

In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from the provided image
data = {
    "Year": [2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019],
    "Jan": [3.3, 6.5, 0.5, 7.5, -6.9, -0.1, 4.6, None],
    "Feb": [2.8, -4.8, 12.6, -5.4, 1.0, 3.2, -5.0, None],
    "Mar": [-10.7, 1.5, 7.0, 7.8, 5.3, 4.4, -7.9, None],
    "Apr": [6.1, 5.5, -5.4, 1.2, -10.2, 4.3, 13.6, None],
    "May": [None, 4.6, 6.8, -4.4, -4.1, -1.1, 4.9, None],
    "Jun": [None, 2.3, -0.3, 1.3, -12.8, -2.9, 3.0, None],
    "Jul": [None, 2.5, 2.5, 2.9, 7.3, 3.5, 14.3, None],
    "Aug": [None, 0.8, -0.2, -3.7, -6.8, 3.1, 2.9, None],
    "Sep": [None, 7.8, 4.2, -5.7, -7.1, -5.9, -5.9, None],
    "Oct": [None, 1.2, 2.6, 7.5, 2.7, 11.1, 0.9, 2.4],
    "Nov": [None, -1.1, 8.3, 7.1, 4.9, -3.5, 5.5, -2.7],
    "Dec": [None, 1.7, -2.7, 3.0, -1.2, 1.8, 14.2, 2.5]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out future months of 2026 and pre-launch months of 2019
)

# Convert return values from percentages to floating fractions (e.g., 3.3 -> 0.033)
df_final["Return"] = df_final["Return"] / 100

# Save output to file or display
print(df_final)

         Return
Period         
2019-10   0.024
2019-11  -0.027
2019-12   0.025
2020-01   0.046
2020-02  -0.050
...         ...
2025-12   0.017
2026-01   0.033
2026-02   0.028
2026-03  -0.107
2026-04   0.061

[79 rows x 1 columns]


In [2]:
df_final.to_csv('./hf_returns/noster.csv')

In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from the provided image
data = {
    "Year": [2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017, 2016],
    "Jan": [2.4, 1.8, 1.3, -0.1, 0.4, -0.4, 0.1, 2.8, 3.6, 2.3, None],
    "Feb": [-0.7, -1.0, 1.4, -0.2, 0.9, 3.8, 1.6, 0.5, 1.0, -1.6, None],
    "Mar": [-0.1, -1.4, 3.1, 0.2, 0.6, -0.3, -4.3, 1.9, -0.4, 3.3, None],
    "Apr": [3.3, 2.6, 0.3, 0.3, 1.0, 1.3, 2.5, 0.5, 3.2, 0.0, None],
    "May": [None, 1.0, 1.6, 0.0, -0.7, -0.2, 2.7, 2.9, 2.9, -1.9, -2.2],
    "Jun": [None, 2.7, 2.6, -0.2, 0.9, 0.0, 1.6, 0.2, 0.3, -0.1, -3.5],
    "Jul": [None, 1.6, 0.3, 0.0, -0.1, -0.2, 1.7, 1.8, 0.0, 0.6, 1.1],
    "Aug": [None, 1.0, 0.4, 0.5, 0.0, 0.5, 0.4, 0.9, 1.4, 3.2, 1.2],
    "Sep": [None, 1.3, 0.3, 0.3, 0.3, 2.2, 0.4, -1.0, 1.0, 0.8, 3.8],
    "Oct": [None, 1.7, 3.7, 1.2, -0.9, 0.0, 1.1, -0.3, -2.2, 1.2, -2.4],
    "Nov": [None, 1.7, 2.1, 1.9, -0.4, -0.1, 1.4, 1.3, -3.0, -0.8, -1.1],
    "Dec": [None, 2.7, 1.7, 0.8, 1.3, 0.4, 4.4, 3.0, 0.6, 2.9, -0.4]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out future months of 2026 and pre-launch months of 2016
)

# Convert return values from percentages to floating fractions (e.g., 2.4 -> 0.024)
df_final["Return"] = df_final["Return"] / 100

# Save output to file or display
print(df_final)

         Return
Period         
2016-05  -0.022
2016-06  -0.035
2016-07   0.011
2016-08   0.012
2016-09   0.038
...         ...
2025-12   0.027
2026-01   0.024
2026-02  -0.007
2026-03  -0.001
2026-04   0.033

[120 rows x 1 columns]


In [2]:
df_final.to_csv('./hf_returns/schonfeld_equity.csv')

In [3]:
import pandas as pd

# 1. Reconstruct the raw grid data from the provided image
data = {
    "Year": [2026, 2025, 2024, 2023, 2022, 2021, 2020],
    "Jan": [5.46, 3.78, 3.84, 5.52, 1.37, 1.78, None],
    "Feb": [2.18, -0.38, 3.52, 0.04, -0.62, 6.81, None],
    "Mar": [-0.94, 0.02, 5.41, -4.55, 4.75, 3.70, None],
    "Apr": [1.32, -4.79, 3.58, 1.73, -0.18, 1.78, -2.68],
    "May": [None, -3.80, -0.33, 7.03, 2.11, 3.71, 6.24],
    "Jun": [None, 1.99, 0.95, 5.74, -3.60, 5.20, 4.93],
    "Jul": [None, 0.86, 3.47, 3.56, -0.35, -0.96, 5.11],
    "Aug": [None, 1.53, 1.18, 0.90, 2.24, 0.63, 2.82],
    "Sep": [None, 0.22, 2.19, 2.02, 0.85, 4.86, -3.17],
    "Oct": [None, 6.38, 3.89, 2.00, -2.97, 3.65, -5.52],
    "Nov": [None, 0.51, 2.41, 3.20, 3.20, -6.45, 13.11],
    "Dec": [None, 2.62, 0.10, 1.74, 2.73, 3.73, 10.41]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out future months of 2026 and pre-launch months of 2020
)

# Convert return values from percentages to floating fractions (e.g., 5.46 -> 0.0546)
df_final["Return"] = df_final["Return"] / 100

# Save output to file or display
print(df_final)

         Return
Period         
2020-04 -0.0268
2020-05  0.0624
2020-06  0.0493
2020-07  0.0511
2020-08  0.0282
...         ...
2025-12  0.0262
2026-01  0.0546
2026-02  0.0218
2026-03 -0.0094
2026-04  0.0132

[73 rows x 1 columns]


In [4]:
df_final.to_csv('./hf_returns/shiprock.csv')

In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from the provided image
data = {
    "Year": [2026, 2025, 2024, 2023, 2022, 2021],
    "Jan": [-0.91, 0.11, 0.80, 2.81, -0.34, None],
    "Feb": [2.56, 4.74, -2.49, -4.14, 2.10, None],
    "Mar": [1.76, 3.32, 2.13, -0.67, 3.21, None],
    "Apr": [-0.48, 6.05, 0.80, 2.54, 0.31, None],
    "May": [None, 0.73, 3.51, 1.33, 1.41, None],
    "Jun": [None, 3.35, 7.06, 2.93, -0.85, 0.75],
    "Jul": [None, 1.40, 6.97, 4.91, 0.16, -0.30],
    "Aug": [None, 2.68, -2.16, 0.57, -0.86, -0.03],
    "Sep": [None, -0.09, 6.60, -0.92, 0.21, 2.51],
    "Oct": [None, 2.99, -3.02, 0.02, 3.71, -0.68],
    "Nov": [None, 2.57, 4.63, 7.51, 0.61, 0.50],
    "Dec": [None, -1.21, -2.93, 2.63, 0.65, 0.48]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out future months of 2026 and pre-launch months of 2021
)

# Convert return values from percentages to floating fractions (e.g., -0.91 -> -0.0091)
df_final["Return"] = df_final["Return"] / 100

# Save output to file or display
print(df_final)

         Return
Period         
2021-06  0.0075
2021-07 -0.0030
2021-08 -0.0003
2021-09  0.0251
2021-10 -0.0068
2021-11  0.0050
2021-12  0.0048
2022-01 -0.0034
2022-02  0.0210
2022-03  0.0321
2022-04  0.0031
2022-05  0.0141
2022-06 -0.0085
2022-07  0.0016
2022-08 -0.0086
2022-09  0.0021
2022-10  0.0371
2022-11  0.0061
2022-12  0.0065
2023-01  0.0281
2023-02 -0.0414
2023-03 -0.0067
2023-04  0.0254
2023-05  0.0133
2023-06  0.0293
2023-07  0.0491
2023-08  0.0057
2023-09 -0.0092
2023-10  0.0002
2023-11  0.0751
2023-12  0.0263
2024-01  0.0080
2024-02 -0.0249
2024-03  0.0213
2024-04  0.0080
2024-05  0.0351
2024-06  0.0706
2024-07  0.0697
2024-08 -0.0216
2024-09  0.0660
2024-10 -0.0302
2024-11  0.0463
2024-12 -0.0293
2025-01  0.0011
2025-02  0.0474
2025-03  0.0332
2025-04  0.0605
2025-05  0.0073
2025-06  0.0335
2025-07  0.0140
2025-08  0.0268
2025-09 -0.0009
2025-10  0.0299
2025-11  0.0257
2025-12 -0.0121
2026-01 -0.0091
2026-02  0.0256
2026-03  0.0176
2026-04 -0.0048


In [2]:
df_final.to_csv('./hf_returns/r_squared.csv')

In [3]:
import pandas as pd

# 1. Reconstruct the raw grid data from the provided image
data = {
    "Year": [2026, 2025, 2024, 2023],
    "Jan": [11.8, 8.9, 4.0, None],
    "Feb": [6.7, -2.2, 0.1, None],
    "Mar": [-9.2, -3.9, -0.8, None],
    "Apr": [-0.4, 5.1, 0.9, None],
    "May": [2.1, 3.5, 2.9, None],
    "Jun": [None, 3.5, 8.1, None],
    "Jul": [None, 1.9, -7.1, None],
    "Aug": [None, 0.0, 2.8, None],
    "Sep": [None, 6.8, 2.6, 0.6],
    "Oct": [None, 4.1, -1.7, 0.9],
    "Nov": [None, 0.5, -2.2, 1.4],
    "Dec": [None, 0.7, 0.7, 2.7]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out future months of 2026 and pre-launch months of 2023
)

# Convert return values from percentages to floating fractions (e.g., 11.8 -> 0.118)
df_final["Return"] = df_final["Return"] / 100

# Save output to file or display
print(df_final)

         Return
Period         
2023-09   0.006
2023-10   0.009
2023-11   0.014
2023-12   0.027
2024-01   0.040
2024-02   0.001
2024-03  -0.008
2024-04   0.009
2024-05   0.029
2024-06   0.081
2024-07  -0.071
2024-08   0.028
2024-09   0.026
2024-10  -0.017
2024-11  -0.022
2024-12   0.007
2025-01   0.089
2025-02  -0.022
2025-03  -0.039
2025-04   0.051
2025-05   0.035
2025-06   0.035
2025-07   0.019
2025-08   0.000
2025-09   0.068
2025-10   0.041
2025-11   0.005
2025-12   0.007
2026-01   0.118
2026-02   0.067
2026-03  -0.092
2026-04  -0.004
2026-05   0.021


In [4]:
df_final.to_csv('./hf_returns/aqua_lake.csv')

In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from the provided image
data = {
    "Year": [2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018],
    "Jan": [3.66, 0.03, 1.85, -1.13, 1.67, 0.40, 0.59, -2.93, None],
    "Feb": [2.62, -4.02, 3.74, 0.83, 3.32, 3.89, -0.47, -0.45, None],
    "Mar": [-1.18, 0.21, 2.77, -3.20, 7.28, 1.23, 4.81, 4.37, None],
    "Apr": [2.25, -4.02, 2.51, 1.98, 4.65, 2.75, -2.00, 1.60, None],
    "May": [None, -2.08, -0.63, 1.62, -0.42, 1.62, -1.82, 1.41, None],
    "Jun": [None, 1.82, -1.80, 3.79, 1.11, -1.33, -1.17, 2.16, None],
    "Jul": [None, -0.57, -1.99, -0.25, -4.76, 0.48, 1.45, 3.06, -1.16],
    "Aug": [None, 2.47, -3.21, -0.51, 3.77, 0.11, 0.26, 7.80, 3.01],
    "Sep": [None, 3.95, 0.19, 2.51, 5.65, 0.69, -2.47, -6.15, -0.79],
    "Oct": [None, 0.12, -3.68, 0.59, -1.53, 3.21, -1.04, -3.68, -3.83],
    "Nov": [None, 1.56, 1.30, -2.82, -4.20, -5.60, 2.65, 0.27, -1.34],
    "Dec": [None, 2.78, 1.90, -0.65, 0.85, 0.76, 5.51, 0.44, 1.88]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out future months of 2026 and pre-launch months of 2018
)

# Convert return values from percentages to floating fractions (e.g., 3.66 -> 0.0366)
df_final["Return"] = df_final["Return"] / 100

# Save output to file or display
print(df_final)

         Return
Period         
2018-07 -0.0116
2018-08  0.0301
2018-09 -0.0079
2018-10 -0.0383
2018-11 -0.0134
...         ...
2025-12  0.0278
2026-01  0.0366
2026-02  0.0262
2026-03 -0.0118
2026-04  0.0225

[94 rows x 1 columns]


In [2]:
df_final.to_csv('./hf_returns/winton.csv')

In [3]:
import pandas as pd

# 1. Reconstruct the raw grid data from the provided image
data = {
    "Year": [2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019],
    "Jan": [3.41, 3.99, 1.15, 0.38, -2.46, 0.65, 2.25, None],
    "Feb": [2.47, -4.05, 0.40, 1.88, 2.65, 5.56, 0.53, None],
    "Mar": [-1.20, -5.35, 8.35, 0.00, -1.82, -5.40, 2.12, None],
    "Apr": [1.65, 1.78, 2.99, -2.21, 0.38, 2.05, 1.77, None],
    "May": [None, 3.73, 0.79, -2.88, -2.57, -1.44, -0.07, None],
    "Jun": [None, 1.78, 2.25, -2.58, 2.48, -0.08, 1.32, None],
    "Jul": [None, 4.53, 0.20, -3.48, 1.41, -2.48, 2.03, None],
    "Aug": [None, -0.64, -2.18, -1.46, 1.95, 0.15, 0.70, None],
    "Sep": [None, 4.35, 1.56, 0.21, 2.10, 3.99, 4.02, -2.57],
    "Oct": [None, 1.14, 5.85, -0.28, 3.35, 2.45, 2.43, 0.03],
    "Nov": [None, 6.15, 7.62, 0.07, -2.68, -1.47, 3.60, -1.86],
    "Dec": [None, 2.89, 1.37, 2.62, 1.96, 0.85, 3.90, -1.11]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out future months of 2026 and pre-launch months of 2019
)

# Convert return values from percentages to floating fractions (e.g., 3.41 -> 0.0341)
df_final["Return"] = df_final["Return"] / 100

# Save output to file or display
print(df_final)

         Return
Period         
2019-09 -0.0257
2019-10  0.0003
2019-11 -0.0186
2019-12 -0.0111
2020-01  0.0225
...         ...
2025-12  0.0289
2026-01  0.0341
2026-02  0.0247
2026-03 -0.0120
2026-04  0.0165

[80 rows x 1 columns]


In [4]:
df_final.to_csv('./hf_returns/cinctive.csv')

In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data using the Month and Percentage Return columns
data = {
    "Year": [2026, 2025, 2024],
    "Jan": [0.63, 2.46, None],
    "Feb": [1.07, 1.02, 1.91],
    "Mar": [3.59, 8.78, -0.94],
    "Apr": [2.43, 0.58, -0.28],
    "May": [1.75, 6.04, 3.98],
    "Jun": [None, 0.33, 3.15],
    "Jul": [None, 0.29, 4.11],
    "Aug": [None, 0.65, 1.56],
    "Sep": [None, 7.98, 0.92],
    "Oct": [None, 2.86, 0.87],
    "Nov": [None, 2.35, -0.89],
    "Dec": [None, -1.74, 2.11]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out future months of 2026 and pre-launch months of 2024
)

# Convert return values from percentages to floating fractions (e.g., 1.91 -> 0.0191)
df_final["Return"] = df_final["Return"] / 100

# Save output to file or display
print(df_final)

         Return
Period         
2024-02  0.0191
2024-03 -0.0094
2024-04 -0.0028
2024-05  0.0398
2024-06  0.0315
2024-07  0.0411
2024-08  0.0156
2024-09  0.0092
2024-10  0.0087
2024-11 -0.0089
2024-12  0.0211
2025-01  0.0246
2025-02  0.0102
2025-03  0.0878
2025-04  0.0058
2025-05  0.0604
2025-06  0.0033
2025-07  0.0029
2025-08  0.0065
2025-09  0.0798
2025-10  0.0286
2025-11  0.0235
2025-12 -0.0174
2026-01  0.0063
2026-02  0.0107
2026-03  0.0359
2026-04  0.0243
2026-05  0.0175


In [2]:
df_final.to_csv('./hf_returns/grasshoper.csv')

In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from Screenshot 2026-06-30 at 09.32.56.png
data = {
    "Year": [2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019],
    "Jan": [-1.5, -3.1, 2.7, 3.2, -2.8, 5.2, 0.6, None],
    "Feb": [7.0, 0.6, 5.4, 0.4, 1.5, 4.1, -4.0, None],
    "Mar": [-0.6, -4.6, 2.1, 3.1, 2.4, -1.2, -4.3, 2.4],
    "Apr": [2.1, 8.5, -5.0, 3.7, -2.7, 5.9, 10.3, 3.4],
    "May": [None, -5.6, -0.2, 2.7, -3.1, 11.3, 6.4, 1.7],
    "Jun": [None, 2.9, -0.2, 0.5, -0.1, 4.5, 0.6, 0.9],
    "Jul": [None, 17.9, 5.4, 8.4, 7.1, -1.2, -1.1, 2.4],
    "Aug": [None, 5.1, 1.7, 1.7, 1.6, 2.3, -0.9, 2.0],
    "Sep": [None, 11.7, 2.0, -0.3, 1.8, 2.3, 4.8, 1.8],
    "Oct": [None, 13.3, 1.7, -6.7, 8.0, 5.1, 3.8, 1.5],
    "Nov": [None, 4.2, 1.0, 4.3, 10.9, -2.9, 5.0, 3.2],
    "Dec": [None, 1.7, -3.9, 8.7, 13.3, 0.8, 5.7, 3.3]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
# Convert month abbreviations to numeric strings (e.g., 'Jan' -> '01')
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Drops non-existent months (Jan/Feb 2019 and May-Dec 2026)
)

# Convert return values from percentages to floating fractions (e.g., -1.5 -> -0.015)
df_final["Return"] = df_final["Return"] / 100

# Save output to file
df_final

,Return
Period,
2019-03,0.024
2019-04,0.034
2019-05,0.017
2019-06,0.009
2019-07,0.024
...,...
2025-12,0.017
2026-01,-0.015
2026-02,0.070


In [2]:
df_final.to_csv('./hf_returns/adar1.csv')

In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from the screenshot
# Note: Negative returns from the image are represented with a minus sign (-)
data = {
    "Year": [2026, 2025, 2024, 2023, 2022, 2021],
    "Jan": [-0.63, -4.05, -2.18, 4.30, 3.79, None],
    "Feb": [2.20, 0.35, 6.33, 6.77, -3.12, 4.98],
    "Mar": [2.41, 2.88, 2.51, 6.51, 5.21, -0.18],
    "Apr": [0.80, 3.04, 2.60, 7.27, 8.79, -0.25],
    "May": [-0.48, 1.33, -3.55, 0.95, 1.55, 6.93],
    "Jun": [None, 2.96, -3.33, -2.66, 5.93, 2.41],
    "Jul": [None, 0.63, 2.24, 0.20, -4.23, 3.99],
    "Aug": [None, -2.07, 3.38, 0.80, 6.36, -1.14],
    "Sep": [None, 1.88, 9.10, -1.30, 1.23, 6.38],
    "Oct": [None, 2.44, 3.19, 0.85, 7.55, -0.41],
    "Nov": [None, 1.79, -1.30, -0.03, -0.08, -1.65],
    "Dec": [None, 2.95, 2.90, 0.86, -4.55, 8.34]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out the missing months (e.g., late 2026, Jan 2021)
)

# Convert return values from percentages to floating fractions (e.g., 2.20 -> 0.022)
df_final["Return"] = df_final["Return"] / 100

# Display the final chronological series
print(df_final)

         Return
Period         
2021-02  0.0498
2021-03 -0.0018
2021-04 -0.0025
2021-05  0.0693
2021-06  0.0241
...         ...
2026-01 -0.0063
2026-02  0.0220
2026-03  0.0241
2026-04  0.0080
2026-05 -0.0048

[64 rows x 1 columns]


In [2]:
df_final.to_csv('./hf_returns/edge_capital.csv')

In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from the new screenshot
# Note: Negative returns from the image are represented with a minus sign (-)
data = {
    "Year": [2026, 2025, 2024, 2023, 2022, 2021],
    "Jan": [2.13, 3.96, -0.36, 4.15, 0.31, 2.14],
    "Feb": [-2.15, 2.15, -2.28, 1.43, 6.47, 1.33],
    "Mar": [1.35, 2.75, 0.56, 3.13, 6.60, 12.68],
    "Apr": [0.99, 9.74, 3.11, 1.84, 9.36, 0.93],
    "May": [None, 1.72, 0.80, 2.38, 7.34, 0.71],
    "Jun": [None, 0.37, -0.09, 1.15, 3.63, -3.14],
    "Jul": [None, 3.77, 3.77, 4.53, -0.10, 3.05],
    "Aug": [None, 1.99, 2.73, 4.28, 7.36, 1.68],
    "Sep": [None, -1.06, -0.70, -1.14, 2.89, 2.62],
    "Oct": [None, 1.64, 0.65, 2.04, 6.51, -1.35],
    "Nov": [None, 3.66, 4.52, 4.12, 3.40, 0.94],
    "Dec": [None, 8.62, 5.78, 2.39, 6.63, 0.57]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out the missing future months of 2026
)

# Convert return values from percentages to floating fractions (e.g., 2.13 -> 0.0213)
df_final["Return"] = df_final["Return"] / 100

# Display the final chronological series
print(df_final)

         Return
Period         
2021-01  0.0214
2021-02  0.0133
2021-03  0.1268
2021-04  0.0093
2021-05  0.0071
...         ...
2025-12  0.0862
2026-01  0.0213
2026-02 -0.0215
2026-03  0.0135
2026-04  0.0099

[64 rows x 1 columns]


In [2]:
df_final.to_csv('./hf_returns/26_miles.csv')

In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from the new screenshot
# Note: Negative returns from the image are represented with a minus sign (-)
data = {
    "Year": [2026, 2025, 2024, 2023],
    "Jan": [-0.6, 3.8, 2.7, None],
    "Feb": [-1.4, -0.1, 4.9, None],
    "Mar": [-4.8, -6.1, 4.2, None],
    "Apr": [9.8, 2.5, -3.3, None],
    "May": [10.3, 15.5, 5.0, None],
    "Jun": [None, 6.0, -1.6, None],
    "Jul": [None, 1.2, -0.9, None],
    "Aug": [None, 2.3, 6.6, -2.2],
    "Sep": [None, 2.8, 4.2, -3.1],
    "Oct": [None, -3.2, 6.2, 0.2],
    "Nov": [None, -1.1, 17.1, 6.2],
    "Dec": [None, -1.0, -4.1, 1.7]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out the missing months
)

# Convert return values from percentages to floating fractions (e.g., 9.8 -> 0.098)
df_final["Return"] = df_final["Return"] / 100

# Display the final chronological series
print(df_final)

         Return
Period         
2023-08  -0.022
2023-09  -0.031
2023-10   0.002
2023-11   0.062
2023-12   0.017
2024-01   0.027
2024-02   0.049
2024-03   0.042
2024-04  -0.033
2024-05   0.050
2024-06  -0.016
2024-07  -0.009
2024-08   0.066
2024-09   0.042
2024-10   0.062
2024-11   0.171
2024-12  -0.041
2025-01   0.038
2025-02  -0.001
2025-03  -0.061
2025-04   0.025
2025-05   0.155
2025-06   0.060
2025-07   0.012
2025-08   0.023
2025-09   0.028
2025-10  -0.032
2025-11  -0.011
2025-12  -0.010
2026-01  -0.006
2026-02  -0.014
2026-03  -0.048
2026-04   0.098
2026-05   0.103


In [2]:
df_final.to_csv('./hf_returns/floating.csv')

In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from the screenshot
# Note: Negative returns are represented with a minus sign (-), and missing data as None
data = {
    "Year": [
        2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017, 
        2016, 2015, 2014, 2013, 2012, 2011, 2010, 2009, 2008, 2007, 2006
    ],
    "Jan": [1.71, 3.98, 0.42, 7.10, 0.63, 3.64, -4.74, 5.75, 0.74, 3.65, -8.37, 6.03, -0.46, 2.82, 8.76, 1.45, -1.87, -5.69, -10.11, 1.58, None],
    "Feb": [1.51, 3.57, 2.45, 4.39, -4.82, 4.03, -6.37, 2.73, -3.33, 0.75, 3.80, 3.74, 6.24, 2.99, 8.93, 1.07, 0.45, -4.18, 0.45, 2.60, None],
    "Mar": [-7.35, -5.92, 3.06, -1.60, -0.70, 6.66, -20.10, -2.29, -2.54, 3.03, 6.37, 2.22, 1.35, -1.47, -1.73, -1.21, 8.42, 3.86, -0.08, 5.15, None],
    "Apr": [5.99, 0.42, -1.08, -1.27, 0.46, 1.30, 10.24, 8.04, 4.03, 3.36, 5.09, -0.23, 2.61, 0.63, -1.35, 6.33, 6.19, 12.72, 3.56, 7.82, None],
    "May": [5.90, 4.03, 6.73, -2.05, 0.14, -0.08, 3.38, -8.47, 1.21, 0.08, 1.35, 0.15, 1.74, 5.31, -6.66, -2.22, -3.42, 0.16, 0.69, 1.91, None],
    "Jun": [None, 0.88, -3.83, 6.34, -8.87, 0.24, 5.23, 5.23, -0.81, -0.49, -8.62, -4.43, -1.19, -7.91, -0.10, -1.25, 0.33, 0.10, -10.38, 0.16, None],
    "Jul": [None, 2.21, 1.07, 2.34, 9.27, 1.22, 6.25, -3.66, 3.28, -3.42, 9.41, 2.86, -4.16, 4.76, 5.12, -5.14, 5.55, 13.35, -3.20, -2.47, 2.02],
    "Aug": [None, 4.35, 0.14, -0.57, -1.78, 1.61, 3.75, -4.47, -1.25, -1.55, 2.37, -8.55, 0.12, 1.93, 3.11, -14.84, -2.88, 2.54, 0.87, -1.24, 8.92],
    "Sep": [None, 1.34, 1.47, -0.67, -5.61, -2.81, -3.86, 5.89, 1.43, 5.23, 2.34, -11.67, -2.46, 3.74, 1.96, -6.06, 13.17, 2.86, -9.15, 3.45, 6.25],
    "Oct": [None, 3.75, 0.22, -4.13, 7.47, 0.45, -2.40, 3.67, -7.62, 1.91, -1.79, 8.57, -0.10, 2.16, -1.37, 10.67, 1.76, 1.32, -11.23, 3.13, 4.95],
    "Nov": [None, 0.43, 3.46, 8.92, 3.85, 0.33, 17.79, 2.65, -4.20, -2.13, 2.75, 4.90, 1.64, 0.93, 3.36, 0.40, 3.20, 2.39, -0.81, -5.15, 0.52],
    "Dec": [None, 4.11, -0.20, 7.53, -2.15, 5.37, 3.29, 3.71, -6.28, 4.28, 4.72, -6.99, -0.26, -0.17, 1.40, 2.76, 6.29, 3.15, 2.12, 1.52, 6.13]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out the missing months (Jan-Jun 2006 & Jun-Dec 2026)
)

# Convert return values from percentages to floating fractions (e.g., 1.71 -> 0.0171)
df_final["Return"] = df_final["Return"] / 100

# Display the final chronological series
print(df_final)

         Return
Period         
2006-07  0.0202
2006-08  0.0892
2006-09  0.0625
2006-10  0.0495
2006-11  0.0052
...         ...
2026-01  0.0171
2026-02  0.0151
2026-03 -0.0735
2026-04  0.0599
2026-05  0.0590

[239 rows x 1 columns]


In [2]:
df_final.to_csv('./hf_returns/cevian.csv')

In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from the screenshot
# Note: Negative returns are represented with a minus sign (-), and missing data as None
data = {
    "Year": [
        2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017, 
        2016, 2015, 2014, 2013, 2012, 2011, 2010, 2009
    ],
    "Jan": [3.66, 3.52, 3.60, 0.20, 0.04, -6.97, 0.00, 0.52, 2.69, 3.20, 2.33, 2.80, -0.39, 1.47, -4.14, -3.66, 1.47, None],
    "Feb": [5.79, 0.16, 0.87, -2.50, 3.20, 4.64, 0.87, -1.46, 0.36, 1.74, -1.88, -2.82, 0.64, 2.17, 0.50, -0.61, -0.53, None],
    "Mar": [-1.56, -2.08, 1.35, 5.02, -0.13, -4.86, -0.18, 2.18, -0.02, 1.87, -3.45, -0.14, -1.49, 5.19, 2.07, 0.53, 2.28, 3.34],
    "Apr": [3.68, -0.68, -0.39, 0.33, -1.56, 0.61, 1.93, -0.14, -1.87, 1.84, 0.39, -4.80, -1.25, -0.14, 2.74, 2.90, -1.08, 6.59],
    "May": [None, -0.55, 0.21, 2.55, 0.42, -0.73, 1.76, 1.46, 3.33, 2.08, 2.81, 2.91, 1.68, 0.32, 2.22, -0.31, -3.66, 8.90],
    "Jun": [None, 3.38, 2.22, -1.39, -0.78, -0.05, -0.54, 0.05, -1.60, -1.43, 3.07, -0.03, -0.22, 2.50, 0.61, 1.15, 2.01, -1.36],
    "Jul": [None, 0.82, 1.51, 1.01, -0.38, -0.10, 1.35, 0.13, 0.38, -1.57, 0.67, 2.46, -0.92, -0.01, 3.25, 5.16, 1.56, 1.53],
    "Aug": [None, 0.97, -0.86, 1.16, 3.70, 2.61, 3.43, 2.94, 0.48, 0.70, -0.43, -1.29, 2.09, -3.86, 0.38, 10.51, 1.53, 7.65],
    "Sep": [None, 1.30, 0.26, 0.13, 0.77, 1.50, -0.31, -2.59, 0.27, -0.68, 1.98, 2.22, 2.32, 0.34, -1.84, 6.57, 2.00, 3.36],
    "Oct": [None, 2.70, -1.31, 3.76, 0.31, 2.99, 0.74, 0.69, -4.08, 3.08, -2.20, -1.24, 2.33, 2.13, -0.92, 1.01, 2.79, -0.65],
    "Nov": [None, 0.69, 1.95, -0.44, -1.02, -4.25, 6.09, 1.08, 1.43, 0.42, -2.74, 4.23, 0.84, 0.84, 2.96, 2.63, -0.93, -0.64],
    "Dec": [None, -0.36, -4.08, -2.22, 0.71, -1.56, 2.15, 1.74, 1.74, 1.07, -0.55, 2.32, 0.60, 1.90, -0.92, -0.83, 0.49, 1.72]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out the missing months (Jan-Feb 2009 & May-Dec 2026)
)

# Convert return values from percentages to floating fractions (e.g., 3.66 -> 0.0366)
df_final["Return"] = df_final["Return"] / 100

# Display the final chronological series
print(df_final)

         Return
Period         
2009-03  0.0334
2009-04  0.0659
2009-05  0.0890
2009-06 -0.0136
2009-07  0.0153
...         ...
2025-12 -0.0036
2026-01  0.0366
2026-02  0.0579
2026-03 -0.0156
2026-04  0.0368

[206 rows x 1 columns]


In [2]:
df_final.to_csv('./hf_returns/mw_opportunities.csv')

In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from the screenshot
# Note: Negative returns are represented with a minus sign (-), and missing data as None
data = {
    "Year": [2026, 2025],
    "Jan": [0.86, None],
    "Feb": [0.72, None],
    "Mar": [1.24, 0.00],
    "Apr": [None, 1.82],
    "May": [None, -0.13],
    "Jun": [None, 1.46],
    "Jul": [None, 2.48],
    "Aug": [None, 3.65],
    "Sep": [None, 0.77],
    "Oct": [None, 0.38],
    "Nov": [None, 11.25],
    "Dec": [None, 0.05]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out the missing months
)

# Convert return values from percentages to floating fractions (e.g., 1.82 -> 0.0182)
df_final["Return"] = df_final["Return"] / 100

# Display the final chronological series
print(df_final)

         Return
Period         
2025-03  0.0000
2025-04  0.0182
2025-05 -0.0013
2025-06  0.0146
2025-07  0.0248
2025-08  0.0365
2025-09  0.0077
2025-10  0.0038
2025-11  0.1125
2025-12  0.0005
2026-01  0.0086
2026-02  0.0072
2026-03  0.0124


In [2]:
df_final.to_csv('./hf_returns/sixth_turn.csv')

In [1]:
import pandas as pd

# 1. Reconstruct the raw grid data from the screenshot
# Note: Negative returns are represented with a minus sign (-), and missing data as None
data = {
    "Year": [2026, 2025, 2024, 2023, 2022],
    "Jan": [-3.08, 6.81, 6.76, -0.37, 0.52],
    "Feb": [-4.78, -0.92, -2.80, 3.15, 2.11],
    "Mar": [2.01, -0.36, 0.64, 1.23, 0.10],
    "Apr": [2.83, -5.84, 19.49, 3.77, 0.00],
    "May": [5.92, 13.41, 8.13, -5.61, 0.00],
    "Jun": [5.43, 4.30, 2.33, 6.95, 0.00],
    "Jul": [None, 6.85, 6.61, 21.02, -5.98],
    "Aug": [None, 6.21, 11.94, 6.24, -0.32],
    "Sep": [None, 11.84, -9.34, -5.42, 2.30],
    "Oct": [None, -3.24, -2.90, -1.58, 0.56],
    "Nov": [None, 1.82, 18.66, 0.25, 3.94],
    "Dec": [None, 0.02, 10.49, 11.69, -0.55]
}

df_raw = pd.DataFrame(data)

# 2. Reshape from wide matrix format to a single chronological series
df_melted = df_raw.melt(id_vars=["Year"], var_name="Month", value_name="Return")

# 3. Build a proper datetime index for the months
month_map = {month: f"{i:02d}" for i, month in enumerate(df_raw.columns[1:], 1)}
df_melted["Month_Num"] = df_melted["Month"].map(month_map)

# Combine Year and Month into a dynamic PeriodIndex (YYYY-MM)
df_melted["Period"] = pd.to_datetime(df_melted["Year"].astype(str) + "-" + df_melted["Month_Num"]).dt.to_period("M")

# 4. Final Clean up
df_final = (
    df_melted.set_index("Period")[["Return"]]
    .sort_index()
    .dropna()  # Filters out the missing months (Jul-Dec 2026)
)

# Convert return values from percentages to floating fractions (e.g., 5.92 -> 0.0592)
df_final["Return"] = df_final["Return"] / 100

# Display the final chronological series
print(df_final)

         Return
Period         
2022-01  0.0052
2022-02  0.0211
2022-03  0.0010
2022-04  0.0000
2022-05  0.0000
2022-06  0.0000
2022-07 -0.0598
2022-08 -0.0032
2022-09  0.0230
2022-10  0.0056
2022-11  0.0394
2022-12 -0.0055
2023-01 -0.0037
2023-02  0.0315
2023-03  0.0123
2023-04  0.0377
2023-05 -0.0561
2023-06  0.0695
2023-07  0.2102
2023-08  0.0624
2023-09 -0.0542
2023-10 -0.0158
2023-11  0.0025
2023-12  0.1169
2024-01  0.0676
2024-02 -0.0280
2024-03  0.0064
2024-04  0.1949
2024-05  0.0813
2024-06  0.0233
2024-07  0.0661
2024-08  0.1194
2024-09 -0.0934
2024-10 -0.0290
2024-11  0.1866
2024-12  0.1049
2025-01  0.0681
2025-02 -0.0092
2025-03 -0.0036
2025-04 -0.0584
2025-05  0.1341
2025-06  0.0430
2025-07  0.0685
2025-08  0.0621
2025-09  0.1184
2025-10 -0.0324
2025-11  0.0182
2025-12  0.0002
2026-01 -0.0308
2026-02 -0.0478
2026-03  0.0201
2026-04  0.0283
2026-05  0.0592
2026-06  0.0543


In [2]:
df_final.to_csv('./hf_returns/spear_digital.csv')